## Using a Triton Inference Server

[Triton Inference Server](https://developer.nvidia.com/triton-inference-server) is an open-source project by NVIDIA for high-performance ML model deployment. In this section, we will practice deploying models using Triton; after you have finished, you should be able to:

-   serve a model using Triton Inference Server with Python backend
-   use dynamic batching to improve performance
-   scale your model to run on multiple GPUs, and/or with multiple instances on the same GPU
-   benchmark the Triton service, and recognize indications of potential problems
-   and use optimized backends

### Anatomy of a Triton model with Python backend

To start, run

``` bash
# runs on node-serve-system
mkdir ~/serve-system-chi/models/
cp -r ~/serve-system-chi/models_staging/food_classifier ~/serve-system-chi/models/
```

to copy [our first configuration](https://github.com/teaching-on-testbeds/serve-system-chi/tree/main/models_staging/food_classifier) into the directory from which Triton will load models.

Our initial implementation serves our food image classifier using PyTorch. Here’s how it works.

In the [Dockerfile](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/docker/Dockerfile.triton), the Triton server is started with the command

``` bash
tritonserver --model-repository=/models
```

where the `/models` directry is organized as follows:

    models/
    └── food_classifier
        ├── 1
        │   ├── food11.pth
        │   └── model.py
        └── config.pbtxt

It includes:

-   a top-level directory whose name is the “model name”
-   a configuration file `config.pbtxt` inside that directory. We’ll look at that shortly.
-   and a subdirectory for each model version. We have model version 1, so we have a subdirectory 1. Inside this directory is a `model.py`, which describes how the model will run.

Let’s [look at the configuration file first](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/models_staging/food_classifier/config.pbtxt). Here are the contents of `config.pbtxt`:

    name: "food_classifier"
    backend: "python"
    max_batch_size: 16
    input [
      {
        name: "INPUT_IMAGE"
        data_type: TYPE_STRING
        dims: [1]
      }
    ]
    output [
      {
        name: "FOOD_LABEL"
        data_type: TYPE_STRING
        dims: [1]
      },
      {
        name: "PROBABILITY"
        data_type: TYPE_FP32
        dims: [1]
      }
    ]
      instance_group [
        {
          count: 1
          kind: KIND_GPU
          gpus: [ 0 ]
        }
    ]

We have defined:

-   a `name`, which must match the directory name
-   a `backend` - we are using the basic [Python backend](https://github.com/triton-inference-server/python_backend). This is a highly flexible backend which allows us to define how our model will run by providing Python code in a `model.py` file.
-   a `max_batch_size` - we have set it to 16, but generally you would set this according to the GPU memory available
-   the `name`, `data_type`, and `dims` (dimensions) of each `input` to the model
-   the `name`, `data_type`, and `dims` (dimensions) of each `output` from the model
-   an `instance_group` with the `count` (number of copies of the model that we want to serve) and details of the device we want to serve it on (we will serve it on GPU 0). Note that to run the model on CPU instead, we could have used

<!-- -->

      instance_group [
        {
          count: 1
          kind: KIND_CPU
        }
      ]

Next, let’s [look at `model.py`](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/models_staging/food_classifier/1/model.py). For a Triton model with Python backend, the `model.py` must define a class named `TritonPythonModel` with at least an `initialize` and `execute` method. Ours has:

-   An `initialize` method to load the model, move it to the device specified in the `args` passed from the Triton server, and put it in inference mode. This will run as soon as Triton starts and loads models from the directory passed to it:

``` python
def initialize(self, args):
        model_dir = os.path.dirname(__file__)
        model_path = os.path.join(model_dir, "food11.pth")
        
        # From args, get info about what device the model is supposed to be on
        instance_kind = args.get("model_instance_kind", "cpu").lower()
        if instance_kind == "gpu":
            device_id = int(args.get("model_instance_device_id", 0))
            torch.cuda.set_device(device_id)
            self.device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device('cpu')

        self.model = torch.load(model_path, map_location=self.device, weights_only=False)
        self.model.to(self.device)
        self.model.eval()

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                    std=[0.229, 0.224, 0.225]),
        ])
        self.classes = np.array([
            "Bread", "Dairy product", "Dessert", "Egg", "Fried food",
            "Meat", "Noodles/Pasta", "Rice", "Seafood", "Soup",
            "Vegetable/Fruit"
        ])
```

-   A `preprocess` method, which will run on each input image that is passed:

``` python
def preprocess(self, image_data):
    if isinstance(image_data, str):
        image_data = base64.b64decode(image_data)

    if isinstance(image_data, bytes):
        image_data = image_data.decode("utf-8")
        image_data = base64.b64decode(image_data)

    image = Image.open(io.BytesIO(image_data)).convert('RGB')

    img_tensor = self.transform(image).unsqueeze(0)
    return img_tensor
```

-   and an `execute`, which will apply to batches of requests sent to this model:

``` python
def execute(self, requests):
    # Gather inputs from all requests
    batched_inputs = []
    for request in requests:
        in_tensor = pb_utils.get_input_tensor_by_name(request, "INPUT_IMAGE")
        input_data_array = in_tensor.as_numpy()  # each assumed to be shape [1]
        # Preprocess each input (resulting in a tensor of shape [1, C, H, W])
        batched_inputs.append(self.preprocess(input_data_array[0, 0]))
    
    # Combine inputs along the batch dimension
    batched_tensor = torch.cat(batched_inputs, dim=0).to(self.device)
    print("BatchSize: ", len(batched_inputs))
    # Run inference once on the full batch
    with torch.no_grad():
        outputs = self.model(batched_tensor)
    
    # Process the outputs and split them for each request
    responses = []
    for i, request in enumerate(requests):
        output = outputs[i:i+1]  # select the i-th output
        prob, predicted_class = torch.max(output, 1)
        predicted_label = self.classes[predicted_class.item()]
        probability = torch.sigmoid(prob).item()
        
        # Create numpy arrays with shape [1, 1] for consistency.
        out_label_np = np.array([[predicted_label]], dtype=object)
        out_prob_np = np.array([[probability]], dtype=np.float32)
        
        out_tensor_label = pb_utils.Tensor("FOOD_LABEL", out_label_np)
        out_tensor_prob = pb_utils.Tensor("PROBABILITY", out_prob_np)
        
        inference_response = pb_utils.InferenceResponse(
            output_tensors=[out_tensor_label, out_tensor_prob])
        responses.append(inference_response)
    
    return responses
```

Finally, now that we understand how the server works, let’s [look at how the Flask app sends requests to it](https://github.com/teaching-on-testbeds/gourmetgram/blob/triton/app.py). Inside the Flask app, we now have a function which is called whenever there is a new image uploaded to `predict` or `test`, which sends the image to the Triton server:

``` python
def request_triton(image_path):
    try:
        # Connect to Triton server
        triton_client = httpclient.InferenceServerClient(url=TRITON_SERVER_URL)

        # Prepare inputs and outputs
        with open(image_path, 'rb') as f:
            image_bytes = f.read()

        inputs = []
        inputs.append(httpclient.InferInput("INPUT_IMAGE", [1, 1], "BYTES"))

        encoded_str =  base64.b64encode(image_bytes).decode("utf-8")
        input_data = np.array([[encoded_str]], dtype=object)
        inputs[0].set_data_from_numpy(input_data)

        outputs = []
        outputs.append(httpclient.InferRequestedOutput("FOOD_LABEL", binary_data=False))
        outputs.append(httpclient.InferRequestedOutput("PROBABILITY", binary_data=False))

        # Run inference
        results = triton_client.infer(model_name=FOOD11_MODEL_NAME, inputs=inputs, outputs=outputs)

        predicted_class = results.as_numpy("FOOD_LABEL")[0,0]
        probability = results.as_numpy("PROBABILITY")[0,0]

        return predicted_class, probability

    except Exception as e:
        print(f"Error during inference: {e}")  
        return None, None  
```

### Bring up containers

To start, run

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up -d
```

This uses a [Docker Compose configuration](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/docker/docker-compose-triton.yaml) to bring up three containers:

-   one container with NVIDIA Triton Server, with the host’s GPUs passed to the container, and with the `models` directory (containing the model and its configuration) passed as a bind mount
-   one container that hosts the Flask app, which will serve the user interface and send inference requests to the Triton server
-   one Jupyter container with the Triton client installed, for us to conduct a performance evaluation of the Triton server

Watch the logs from the Triton server as it starts up:

``` bash
# runs on node-serve-system
docker logs triton_server -f
```

Once the Triton server starts up, you should see something like

    +--------------------------+---------+--------+
    | Model                    | Version | Status |
    +--------------------------+---------+--------+
    | food_classifier | 1       | READY  |
    +--------------------------+---------+--------+

and then some additional output. Near the end, you will see

    "Started GRPCInferenceService at 0.0.0.0:8001"
    "Started HTTPService at 0.0.0.0:8000"
    "Started Metrics Service at 0.0.0.0:8002"

(and then some messages about not getting GPU power consumption, which is fine and not a concern.)

You can use Ctrl+C to stop watching the logs once you see this output.

Let’s test this service. In a browser, run

    http://A.B.C.D

but substitute the floating IP assigned to your instance, to access the Flask app. Upload an image and press “Submit” to get its class label.

Finally, check the logs of the Jupyter container:

``` bash
# runs on node-serve-system
docker logs jupyter
```

and look for a line like

    http://127.0.0.1:8888/lab?token=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

Paste this into a browser tab, but in place of 127.0.0.1, substitute the floating IP assigned to your instance, to open the Jupyter notebook interface that is running *on your compute instance*.

Then, in the file browser on the left side, open the “work” directory and then click on the `triton.ipynb` notebook to continue.

Meanwhile, on the host, run

``` bash
# runs on node-serve-system
nvtop
```

to monitor GPU usage - we will refer back to this a few times as we run through the rest of this notebook.

### Serving a PyTorch model

The Triton client comes with a performance analyzer, which we can use to send requests to the server and get some statistics back. Let’s try it:

In [1]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 1 

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 1035
    Throughput: 57.1653 infer/sec
    Avg latency: 17393 usec (standard deviation 324 usec)
    p50 latency: 17333 usec
    p90 latency: 17563 usec
    p95 latency: 17797 usec
    p99 latency: 18695 usec
    Avg HTTP time: 17386 usec (send/recv 210 usec + response wait 17176 usec)
  Server: 
    Inference count: 1035
    Execution count: 1035
    Successful request count: 1035
    Avg request latency: 16593 usec (overhead 3 usec + queue 67 usec + compute input 55 usec + compute infer 16383 usec + compute output 83 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 57.1653 infer/sec, latency 17393 us

Make a note of the line showing the total average request latency, and the breakdown including:

-   `queue`, the queuing delay
-   and `compute infer`, the inference delay

<!--

    Avg request latency: 18689 usec (overhead 2 usec + queue 22 usec + compute input 44 usec + compute infer 18570 usec + compute output 49 usec)

Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 51.549 infer/sec, latency 19311 usec

-->

Let’s further exercise this service. In the command above, a single client sends continuous requests to the server - each time a response is returned, a new request is generated. Now, let’s configure **8** concurrent clients, each sending continuous requests - as soon as any client gets a response, it sends a new request:

In [18]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 1 --request-distribution=poisson --concurrency-range 1

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 1030
    Throughput: 56.9957 infer/sec
    Avg latency: 17463 usec (standard deviation 926 usec)
    p50 latency: 17317 usec
    p90 latency: 17736 usec
    p95 latency: 17984 usec
    p99 latency: 18808 usec
    Avg HTTP time: 17456 usec (send/recv 207 usec + response wait 17249 usec)
  Server: 
    Inference count: 1030
    Execution count: 1030
    Successful request count: 1030
    Avg request latency: 16677 usec (overhead 2 usec + queue 65 usec + compute input 54 usec + compute infer 16475 usec + compute output 80 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 56.9957 infer/sec, latency 17463 us

<!-- 

    Avg request latency: 151375 usec (overhead 3 usec + queue 132341 usec + compute input 59 usec + compute infer 18922 usec + compute output 49 usec)

Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 52.3786 infer/sec, latency 151983 usec

-->

While the inference time (`compute infer`) remains low, the overall system latency is high because of `queue` delay. Only one sample is processed at a time, and other samples have to wait in a queue for their turn. Here, since there are 8 concurrent clients sending continuous requests, the delay is approximately 8x the inference delay. With more concurrent requests, the queuing delay would grow even larger:

In [30]:
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 1 --request-distribution=poisson --concurrency-range 1:8 --collect-metrics -f metrics_output.csv

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Concurrency limit: 8 concurrent requests
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 1034
    Throughput: 57.1949 infer/sec
    Avg latency: 17390 usec (standard deviation 1336 usec)
    p50 latency: 17185 usec
    p90 latency: 17526 usec
    p95 latency: 18181 usec
    p99 latency: 21263 usec
    Avg HTTP time: 17383 usec (send/recv 197 usec + response wait 17186 usec)
  Server: 
    Inference count: 1034
    Execution count: 1034
    Successful request count: 1034
    Avg request latency: 16608 usec (overhead 3 usec + queue 67 usec + compute input 56 usec + compute infer 16401 usec + compute output 79 usec)
  Server Prometheus Metrics: 
    Avg GPU Util

In [33]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 1 --request-distribution=poisson --concurrency-range 1:8 --log-frequency=10 --collect-metrics -f metrics_output_test.csv

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Concurrency limit: 8 concurrent requests
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 931
    Throughput: 51.4832 infer/sec
    Avg latency: 19321 usec (standard deviation 1501 usec)
    p50 latency: 19045 usec
    p90 latency: 19528 usec
    p95 latency: 19992 usec
    p99 latency: 24033 usec
    Avg HTTP time: 19315 usec (send/recv 226 usec + response wait 19089 usec)
  Server: 
    Inference count: 931
    Execution count: 931
    Successful request count: 931
    Avg request latency: 18521 usec (overhead 4 usec + queue 66 usec + compute input 55 usec + compute infer 18315 usec + compute output 81 usec)
  Server Prometheus Metrics: 
    Avg GPU Utilizat

In [34]:
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 1 --request-distribution=poisson --concurrency-range 1:8 --collect-metrics -f metrics_output_test2.csv

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Concurrency limit: 8 concurrent requests
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 932
    Throughput: 51.5819 infer/sec
    Avg latency: 19274 usec (standard deviation 1565 usec)
    p50 latency: 18980 usec
    p90 latency: 19505 usec
    p95 latency: 20193 usec
    p99 latency: 23958 usec
    Avg HTTP time: 19267 usec (send/recv 220 usec + response wait 19047 usec)
  Server: 
    Inference count: 932
    Execution count: 932
    Successful request count: 932
    Avg request latency: 18459 usec (overhead 3 usec + queue 66 usec + compute input 56 usec + compute infer 18254 usec + compute output 78 usec)
  Server Prometheus Metrics: 
    Avg GPU Utilizat

In [ ]:
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 1 --request-distribution=poisson --concurrency-range 1:8 --log-frequency=10 --collect-metrics -f metrics_output_test.csv

In [ ]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 16 --concurrency-range 1

Although the delay is large (over 100 ms), it’s not because of inadequate compute - if you check the `nvtop` display on the host while the test above is running, you will note low GPU utilization! Take a screenshot of the `nvtop` output when this test is running.

We *could* get more throughput without increasing prediction latency, by batching requests:

#Skipping the following part for now

But, that’s not very helpful in a situation when requests come from individual users, one at a time.

### Dynamic batching

Earlier, we noted that our model can achieve higher throughput with low latency by performing inference on batches of input samples, instead of individual samples. However, our client sends requests with individual samples.

To improve performance, we can ask the Triton server to batch incoming requests whenever possible, and send them through the server together instead of a sequence. In other words, if the server is ready to handle the next request, and it finds four requests waiting in the queue, it should serve those four as a batch instead of just taking the next request in line.

In [79]:
# runs inside Jupyter container
curl http://triton_server:8000/v2/models/food_classifier/versions/1/stats

{"model_stats":[{"name":"food_classifier","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


Then, run the benchmark:

In [13]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 1 --concurrency-range 8

 Successfully read data for 1 stream/streams with 1 step/steps.
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 8
  Client: 
    Request count: 1008
    Throughput: 55.6786 infer/sec
    Avg latency: 143619 usec (standard deviation 1925 usec)
    p50 latency: 144246 usec
    p90 latency: 145362 usec
    p95 latency: 145835 usec
    p99 latency: 148935 usec
    Avg HTTP time: 143613 usec (send/recv 176 usec + response wait 143437 usec)
  Server: 
    Inference count: 1008
    Execution count: 1008
    Successful request count: 1008
    Avg request latency: 142989 usec (overhead 3 usec + queue 125066 usec + compute input 62 usec + compute infer 17809 usec + compute output 48 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 55.6786 infer/sec, lat

<!--

    Avg request latency: 100423 usec (overhead 6 usec + queue 44892 usec + compute input 197 usec + compute infer 55111 usec + compute output 216 usec)

Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 78.6276 infer/sec, latency 101232 usec

-->

and get per-batch stats again:

In [14]:
# runs inside Jupyter container
curl http://triton_server:8000/v2/models/food_classifier/versions/1/stats

{"model_stats":[{"name":"food_classifier","version":"1","last_inference":1751477106950,"inference_count":1217,"execution_count":1217,"inference_stats":{"success":{"count":1217,"ns":192651884282},"fail":{"count":0,"ns":0},"queue":{"count":1217,"ns":168445187294},"compute_input":{"count":1217,"ns":76722036},"compute_infer":{"count":1217,"ns":24068869791},"compute_output":{"count":1217,"ns":59199139},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1217,"ns":76722036},"compute_infer":{"count":1217,"ns":24068869791},"compute_output":{"count":1217,"ns":59199139}}],"memory_usage":[]}]}


<!--

{"model_stats":[{"name":"food_classifier","version":"1","last_inference":1741928954242,"inference_count":1436,"execution_count":386,"inference_stats":{"success":{"count":1436,"ns":144129653806},"fail":{"count":0,"ns":0},"queue":{"count":1436,"ns":64542800676},"compute_input":{"count":1436,"ns":283368073},"compute_infer":{"count":1436,"ns":78984688177},"compute_output":{"count":1436,"ns":309635270},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":26,"ns":1754466},"compute_infer":{"count":26,"ns":757012965},"compute_output":{"count":26,"ns":2038319}},{"batch_size":2,"compute_input":{"count":127,"ns":14474588},"compute_infer":{"count":127,"ns":3718519926},"compute_output":{"count":127,"ns":13184875}},{"batch_size":3,"compute_input":{"count":55,"ns":7182962},"compute_infer":{"count":55,"ns":2144383142},"compute_output":{"count":55,"ns":7683505}},{"batch_size":4,"compute_input":{"count":9,"ns":1446080},"compute_infer":{"count":9,"ns":456549788},"compute_output":{"count":9,"ns":1596636}},{"batch_size":5,"compute_input":{"count":73,"ns":14796021},"compute_infer":{"count":73,"ns":4268808423},"compute_output":{"count":73,"ns":16766209}},{"batch_size":6,"compute_input":{"count":82,"ns":19691717},"compute_infer":{"count":82,"ns":5577222019},"compute_output":{"count":82,"ns":22604780}},{"batch_size":7,"compute_input":{"count":14,"ns":4742974},"compute_infer":{"count":14,"ns":1103416079},"compute_output":{"count":14,"ns":4618631}}],"memory_usage":[]}]}

-->

Note that the stats show that some requests were served in batch sizes greater than 1, even though each client sent a single request at a time.

### Scaling up

Another easy way to improve performance is to scale up! Let’s edit the model configuration:

``` bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier/config.pbtxt
```

and change

      instance_group [
        {
          count: 1
          kind: KIND_GPU
          gpus: [ 0 ]
        }
    ]

to run two instances on GPU 0 and two instances on GPU 1:

      instance_group [
        {
          count: 2
          kind: KIND_GPU
          gpus: [ 0 ]
        },
        {
          count: 2
          kind: KIND_GPU
          gpus: [ 1 ]
        }
    ]

Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

and use

``` bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

On the host, run

``` bash
# runs on node-serve-system
nvidia-smi
```

and note that there are two instances of `triton_python_backend` processes running on GPU 0, and two on GPU 1.

Then, benchmark *this* service with increased concurrency:

In [ ]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 1 --concurrency-range 8

<!-- 

    Avg request latency: 40707 usec (overhead 3 usec + queue 7036 usec + compute input 75 usec + compute infer 33514 usec + compute output 78 usec)

Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 192.849 infer/sec, latency 41374 usec

-->

Although there is still some queuing delay (because our degree of concurrency, 8, is still higher than the number of server instances, 4), and the inference time is also increased due to sharing the compute resources, the prediction delay is still on the order of 10s of ms - not over 100ms, like it was previously with concurrency 8!

Also, if you look at the `nvtop` output on the host while running this test, you will observe higher GPU utilization than before (which is good! We want to use the GPU. Underutilization is bad.) (Take a screenshot!) However, we are still not fully utilizing the GPU.

Let’s try increasing the number of instances again. Edit the model configuration:

``` bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier/config.pbtxt
```

and change

      instance_group [
        {
          count: 2
          kind: KIND_GPU
          gpus: [ 0 ]
        },
        {
          count: 2
          kind: KIND_GPU
          gpus: [ 1 ]
        }
    ]

to

      instance_group [
        {
          count: 4
          kind: KIND_GPU
          gpus: [ 0 ]
        },
        {
          count: 4
          kind: KIND_GPU
          gpus: [ 1 ]
        }
    ]

Re-build the container image with this change:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

use

``` bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

Then, re-run our benchmark:

In [ ]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier  --input-data input.json -b 1 --concurrency-range 8

<!--

    Avg request latency: 66737 usec (overhead 2 usec + queue 466 usec + compute input 61 usec + compute infer 66118 usec + compute output 89 usec)

Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 118.688 infer/sec, latency 67559 usec

-->

This makes things worse - our inference time is higher, even though we are still underutilizing the GPU (as seen in `nvtop`) (take a screenshot!).

Our system is not limited by GPU - we are underutilizing the GPU. However, we are being killed by the overhead of the Python backend and our `model.py` implementation.

#Skipped up to here!!

### Serving an ONNX model

The Python backend we have been using is flexible, but not necessarily the most performant. To get better performance, we will use one of the highly optimized backend in Triton. Since we already have an ONNX model, let’s use the ONNX backend.

To serve a model using the ONNX backend, we will create a [directory structure like this](https://github.com/teaching-on-testbeds/serve-system-chi/tree/main/models_staging/food_classifier_onnx):

    food_classifier_onnx/
    ├── 1
    │   └── model.onnx
    └── config.pbtxt

There is no more `model.py` - Triton serves the model directly, we just have to name it `model.onnx`. In [`config.pbtxt`](https://github.com/teaching-on-testbeds/serve-system-chi/blob/main/models_staging/food_classifier_onnx/config.pbtxt), we will specify the backend as `onnxruntime`:

    name: "food_classifier_onnx"
    backend: "onnxruntime"
    max_batch_size: 16
    input [
      {
        name: "input"  # has to match ONNX model's input name
        data_type: TYPE_FP32
        dims: [3, 224, 224]  # has to match ONNX input shape
      }
    ]
    output [
      {
        name: "output"  # has to match ONNX model output name
        data_type: TYPE_FP32  # output is a list of probabilities
        dims: [11]  # 
      }
    ]
      instance_group [
        {
          count: 1
          kind: KIND_GPU
          gpus: [ 0 ]
        }
    ]

Copy this to Triton’s models directory:

``` bash
# runs on node-serve-system
cp -r ~/summer2025/models/food_classifier_onnx ~/exp-chi/models/
```

Re-build the container image with this change:

``` bash
# runs on node-serve-system
docker compose -f ~/exp-chi/docker/docker/docker-compose.yaml build triton_server
```

and then bring the server back up:

``` bash
# runs on node-serve-system
docker compose -f ~/exp-chi/docker/docker/docker-compose.yaml up triton_server --force-recreate -d
```

use

``` bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready. Note that the server will load two models: the original `food_classifier` with Python backend, and the `food_classifier_onnx` model we just added.

Let’s benchmark our service. Our ONNX model won’t accept image bytes directly - it expects images that already have been pre-processed into arrays. So, our benchmark command will be a little bit different:

In [35]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 2400
    Throughput: 125.502 infer/sec
    Avg latency: 7373 usec (standard deviation 868 usec)
    p50 latency: 7388 usec
    p90 latency: 8330 usec
    p95 latency: 8442 usec
    p99 latency: 9236 usec
    Avg HTTP time: 7365 usec (send/recv 520 usec + response wait 6845 usec)
  Server: 
    Inference count: 2400
    Execution count: 2400
    Successful request count: 2400
    Avg request latency: 5285 usec (overhead 35 usec + queue 80 usec + compute input 231 usec + compute infer 4918 usec + compute output 19 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 125.502 infer/sec, latency 7373 usec


In [35]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 2400
    Throughput: 125.502 infer/sec
    Avg latency: 7373 usec (standard deviation 868 usec)
    p50 latency: 7388 usec
    p90 latency: 8330 usec
    p95 latency: 8442 usec
    p99 latency: 9236 usec
    Avg HTTP time: 7365 usec (send/recv 520 usec + response wait 6845 usec)
  Server: 
    Inference count: 2400
    Execution count: 2400
    Successful request count: 2400
    Avg request latency: 5285 usec (overhead 35 usec + queue 80 usec + compute input 231 usec + compute infer 4918 usec + compute output 19 usec)
Inferences/Second vs. Client Average Batch Latency
Concurrency: 1, throughput: 125.502 infer/sec, latency 7373 usec


In [37]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-distribution=poisson --concurrency-range=1:8 --collect-metrics -f metrics_onnx1.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Concurrency limit: 8 concurrent requests
  Using synchronous calls for inference

Request concurrency: 1
  Client: 
    Request count: 2227
    Throughput: 114.747 infer/sec
    Avg latency: 7948 usec (standard deviation 907 usec)
    p50 latency: 7758 usec
    p90 latency: 9213 usec
    p95 latency: 9415 usec
    p99 latency: 9502 usec
    Avg HTTP time: 7939 usec (send/recv 586 usec + response wait 7353 usec)
  Server: 
    Inference count: 2227
    Execution count: 2227
    Successful request count: 2227
    Avg request latency: 5581 usec (overhead 37 usec + queue 82 usec + compute input 238 usec + compute infer 5202 usec + compute output 21 usec)
  Server Prometheus Metrics: 
    Avg GPU Utilization:
      GPU-81207bda-7e38-0495-0510-11595cdbff2c : 0%
      GPU-f

In [ ]:
# Simulation of M/D/1 queue varying λ from 36.5-328.5 considering an average μ=365 (to get values for ρ in [0.1,0.9])

In [78]:
  perf_analyzer -u triton_server:8000 \
  -m food_classifier_onnx \
  -b 1 \
  --shape IMAGE:3,224,224 \
  --request-rate-range 11:99:11 \
  --request-distribution=poisson \
  --collect-metrics \
  -f test_output.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 99 requests per seconds
  Using poisson distribution on request generation
  Using synchronous calls for inference

Request Rate: 11 inference requests per second
  Client: 
    Request count: 161
    Throughput: 8.89096 infer/sec
    Avg latency: 10257 usec (standard deviation 1565 usec)
    p50 latency: 9837 usec
    p90 latency: 10652 usec
    p95 latency: 14947 usec
    p99 latency: 16994 usec
    Avg HTTP time: 10247 usec (send/recv 1062 usec + response wait 9185 usec)
  Server: 
    Inference count: 161
    Execution count: 161
    Successful request count: 161
    Avg request latency: 7253 usec (overhead 44 usec + queue 485 usec + compute input 201 usec + compute infer 6496 usec + compute output 26 usec)
  Server Prometheus Metrics: 
    Av

In [ ]:
#Run above gave an average μ of 20000 => adjusting λ for the test below to actually get ρ values between [0.1,0.9]
#Simulation of M/D/1 queue varying λ from 2000-18000 considering an average μ=365 (to get values for ρ in [0.1,0.9])

In [ ]:
#!/bin/bash

# Estimated service rate μ (inferences/sec)
MU=100

SUMMARY_FILE="rho_summary.csv"

# Write header if file doesn't exist
if [ ! -f "$SUMMARY_FILE" ]; then
  echo "rho,lambda,inferences_per_sec,server_queue,p50_latency" > "$SUMMARY_FILE"
fi

# Loop over rho values
for RHO in 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9; do
  LAMBDA=$(awk "BEGIN {printf \"%d\", $RHO * $MU}")
  OUTFILE="temp_rho_${RHO}.csv"

  echo "▶ Running for ρ=$RHO (λ=$LAMBDA req/sec) at fixed concurrency $FIXED_CONCURRENCY"

  perf_analyzer -u "$SERVER_URL" \
    -m food_classifier_onnx \
    -b 1 \
    --shape IMAGE:3,224,224 \
    --request-rate-range ${LAMBDA}:${LAMBDA}:1 \
    --request-distribution=poisson \
    --collect-metrics \
    -f "$OUTFILE"

  if [ -f "$OUTFILE" ]; then
    LINE=$(tail -n +2 "$OUTFILE" | head -1)
    if [ -n "$LINE" ]; then
      INF_SEC=$(echo "$LINE" | cut -d',' -f2)
      SERVER_QUEUE=$(echo "$LINE" | cut -d',' -f6)
      P50_LATENCY=$(echo "$LINE" | cut -d',' -f12)
      echo "$RHO,$LAMBDA,$INF_SEC,$SERVER_QUEUE,$P50_LATENCY" >> "$SUMMARY_FILE"
      echo "Logged for ρ=$RHO"
    else
      echo "No stable data at ρ=$RHO"
    fi
  else
    echo "Failed to run perf_analyzer for ρ=$RHO"
  fi
done

echo "Done! Summary in $SUMMARY_FILE"


▶ Running for ρ=0.1 (λ=10 req/sec) at fixed concurrency 
*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 10 requests per seconds
  Using poisson distribution on request generation
  Using synchronous calls for inference

Request Rate: 10 inference requests per second
  Client: 
    Request count: 182
    Throughput: 10.0259 infer/sec
    Avg latency: 10016 usec (standard deviation 1013 usec)
    p50 latency: 9949 usec
    p90 latency: 10168 usec
    p95 latency: 10347 usec
    p99 latency: 15253 usec
    Avg HTTP time: 10007 usec (send/recv 1043 usec + response wait 8964 usec)
  Server: 
    Inference count: 182
    Execution count: 182
    Successful request count: 182
    Avg request latency: 7011 usec (overhead 45 usec + queue 234 usec + compute input 218 usec + compute infer 6487 usec + com

In [70]:
perf_analyzer -u triton_server:8000 \
  -m food_classifier_onnx \
  -b 1 \
  --shape IMAGE:3,224,224 \
  --request-rate-range 11:99:11 \
  --request-distribution=poisson \
  --collect-metrics \
  -f test_output.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 99 requests per seconds
  Using poisson distribution on request generation
  Using synchronous calls for inference

Request Rate: 11 inference requests per second
  Client: 
    Request count: 161
    Throughput: 8.89008 infer/sec
    Avg latency: 10213 usec (standard deviation 1569 usec)
    p50 latency: 9825 usec
    p90 latency: 10831 usec
    p95 latency: 14911 usec
    p99 latency: 17069 usec
    Avg HTTP time: 10203 usec (send/recv 1061 usec + response wait 9142 usec)
  Server: 
    Inference count: 161
    Execution count: 161
    Successful request count: 161
    Avg request latency: 7207 usec (overhead 43 usec + queue 476 usec + compute input 202 usec + compute infer 6458 usec + compute output 27 usec)
  Server Prometheus Metrics: 
    Av

This model has much better inference performance than our PyTorch model with Python backend did, in a similar test. Also, if we monitor with `nvtop`, we should see higher GPU utilization while the test is running (which is a good thing!) (Take a screenshot!)

Let’s try scaling *this* model up. Edit the model configuration:

``` bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier_onnx/config.pbtxt
```

and change

      instance_group [
        {
          count: 1
          kind: KIND_GPU
          gpus: [ 0 ]
        }
    ]

to

      instance_group [
        {
          count: 2      
          kind: KIND_GPU
          gpus: [ 0, 1 ]
        }
    ]

Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

and use

``` bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

Then, run our benchmark with higher concurrency. (2 instances on each GPU, because we noticed that a single instance used less than half a GPU.)

Watch the `nvtop` output as you run this test! (Take a screenshot!)

In [ ]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --concurrency-range 8 

<!-- 

    Avg request latency: 3961 usec (overhead 18 usec + queue 697 usec + compute input 97 usec + compute infer 3137 usec + compute output 11 usec)

Inferences/Second vs. Client Average Batch Latency
Concurrency: 8, throughput: 1182.39 infer/sec, latency 6089 usec

-->

This time, we should see that our model is fully utilizing the GPU (that’s good!) And, our inference performance is much better than the PyTorch model with Python backend could achieve with concurrency 8.

Let’s see how we do with even higher concurrency:

In [ ]:
# runs inside Jupyter container
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --concurrency-range 16  

<!-- 



    Avg request latency: 9960 usec (overhead 19 usec + queue 6793 usec + compute input 100 usec + compute infer 3036 usec + compute output 11 usec)

Inferences/Second vs. Client Average Batch Latency
Concurrency: 16, throughput: 1257.15 infer/sec, latency 12025 usec

-->

We still have some queue delay, since the rate at which requests arrive is greater than the service rate of the models. But, we can feel good that we are no longer underutilizing the GPUs!

There’s one more issue we should address: our ONNX model doesn’t directly work with our Flask server now, because the inputs and outputs are different. The ONNX model expects a pre-processed array, and returns a list of class probabilities.

Since the pre-processing and post-processing doesn’t need GPU anyway, we’ll move it to the Flask app.

Edit the Docker compose file:

``` bash
# runs on node-serve-system
nano ~/serve-system-chi/docker/docker-compose-triton.yaml
```

and change

      flask:
        build:
          context: https://github.com/teaching-on-testbeds/gourmetgram.git#triton

to

      flask:
        build:
          context: https://github.com/teaching-on-testbeds/gourmetgram.git#triton_onnx

to use [a version of our Flask app where the pre- and post-processing is built in](https://github.com/teaching-on-testbeds/gourmetgram/blob/triton_onnx/app.py). Also change

          - FOOD11_MODEL_NAME=food_classifier

to

          - FOOD11_MODEL_NAME=food_classifier_onnx

so that our Flask app will send requests to the new ONNX model service.

Then run

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build flask
```

to re-build the container image, and

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up flask --force-recreate -d
```

to restart the Flask container with the new image.

Let’s test this service. In a browser, run

    http://A.B.C.D

but substitute the floating IP assigned to your instance, to access the Flask app. Upload an image and press “Submit” to get its class label.

Then, download this entire notebook for later reference.

## Experiment

In [92]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
--request-distribution=constant --request-rate-range 100

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using synchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1927
    Throughput: 100.003 infer/sec
    Avg latency: 8476 usec (standard deviation 639 usec)
    p50 latency: 8722 usec
    p90 latency: 8888 usec
    p95 latency: 9036 usec
    p99 latency: 9330 usec
    Avg HTTP time: 8467 usec (send/recv 380 usec + response wait 8087 usec)
  Server: 
    Inference count: 1927
    Execution count: 1927
    Successful request count: 1927
    Avg request latency: 6353 usec (overhead 43 usec + queue 97 usec + compute input 232 usec + compute infer 5955 usec + compute output 25 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 100.003 infer/sec, latency 

In [93]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using synchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1832
    Avg send request rate: 99.36 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.77% of the requests were delayed. 
    Throughput: 97.53 infer/sec
    Avg latency: 10952 usec (standard deviation 3851 usec)
    p50 latency: 9494 usec
    p90 latency: 16554 usec
    p95 latency: 18753 usec
    p99 latency: 22897 usec
    Avg HTTP time: 10943 usec (send/recv 742 usec + response wait 10201 usec)
  Server: 
    Inference count: 1832
    Execution count: 1832
    Successful request count: 1832
    Avg request latency: 8080 usec (overhead 37 usec + queue 2816 usec + compute input 

In [94]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 200

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using synchronous calls for inference

Request Rate: 200 inference requests per second
  Client: 
    Request count: 3918
    Avg send request rate: 203.79 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 18.27% of the requests were delayed. 
    Throughput: 196.63 infer/sec
    Avg latency: 9522 usec (standard deviation 3035 usec)
    p50 latency: 9128 usec
    p90 latency: 13364 usec
    p95 latency: 14019 usec
    p99 latency: 16278 usec
    Avg HTTP time: 9513 usec (send/recv 678 usec + response wait 8835 usec)
  Server: 
    Inference count: 3918
    Execution count: 3918
    Successful request count: 3918
    Avg request latency: 6745 usec (overhead 25 usec + queue 3241 usec + compute input 

In [95]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average latency and throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using synchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 8779
    Avg send request rate: 437.15 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 99.92% of the requests were delayed. 
    Throughput: 372.62 infer/sec
    Avg latency: 10006 usec (standard deviation 1103 usec)
    p50 latency: 10286 usec
    p90 latency: 10522 usec
    p95 latency: 10758 usec
    p99 latency: 13163 usec
    Avg HTTP time: 9998 usec (send/recv 984 usec + response wait 9014 usec)
  Server: 
    Inference count: 8780
    Execution count: 8780
    Successful request count: 8780
    Avg request latency: 7278 usec (overhead 18 usec + queue 4609 usec + compute inpu

In [96]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --concurrency-range 4

Usage: perf_analyzer [options]
==== SYNOPSIS ====
 
	--version 
	-m <model name>
	-x <model version>
	--bls-composing-models <string>
	--model-signature-name <model signature name>
	--service-kind <"triton"|"openai"|"tfserving"|"torchserve"|"triton_c_api">
	--endpoint <string>
	-v

I. MEASUREMENT PARAMETERS: 
	--async (-a)
	--sync
	--measurement-interval (-p) <measurement window (in msec)>
	--concurrency-range <start:end:step>
	--periodic-concurrency-range <start:end:step>
	--session-concurrency <session concurrency>
	--request-period <number of responses>
	--request-rate-range <start:end:step>
	--request-distribution <"poisson"|"constant">
	--request-intervals <path to file containing time intervals in microseconds>
	--serial-sequences
	--binary-search
	--num-of-sequences <number of concurrent sequences>
	--latency-threshold (-l) <latency threshold (in msec)>
	--max-threads <thread counts>
	--stability-percentage (-s) <deviation threshold for stable measurement (in percentage)>
	--max

: 99

In [97]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 7629
    Avg send request rate: 510.34 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.82% of the requests were delayed. 
    Throughput: 374.00 infer/sec
    Avg latency: 20344624 usec (standard deviation 20347 usec)
    p50 latency: 20293961 usec
    p90 latency: 22349032 usec
    p95 latency: 22599814 usec
    p99 latency: 22829916 usec
    Avg HTTP time: 20344738 usec (send/recv 13233 usec + response wait 20331505 usec)
  Server: 
    Inference count: 7628
    Execution count: 7628
    Successful request count: 7628
    Avg request latency: 20307088 usec (overhead 19 usec + queue 203044

In [98]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 330 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 330 inference requests per second
  Client: 
    Request count: 7381
    Avg send request rate: 335.79 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.30% of the requests were delayed. 
    Throughput: 331.94 infer/sec
    Avg latency: 27890 usec (standard deviation 33067 usec)
    p50 latency: 14914 usec
    p90 latency: 84995 usec
    p95 latency: 115915 usec
    p99 latency: 136416 usec
    Avg HTTP time: 27616 usec (send/recv 519 usec + response wait 27097 usec)
  Server: 
    Inference count: 7381
    Execution count: 7381
    Successful request count: 7381
    Avg request latency: 24404 usec (overhead 17 usec + queue 21716 usec + compute input 85 

Expected waiting time is
$$\frac{1}{2\mu}\cdot\frac{\rho}{1-\rho}$$

In [116]:
rho_val=(0.2 0.9)
max_service_rate=330
step_size=$(awk -v d="$max_service_rate" 'BEGIN { printf "%.2f", 0.1 * d }')
request_arg=""
for rho in "${rho_val[@]}"; do
    request_rate=$(awk -v v="$rho" -v d="$max_service_rate" 'BEGIN { printf "%.2f", v * d }')
    echo $request_rate
    request_arg+="$request_rate:"
done

request_arg+="$step_size"
echo $request_arg

66.00
297.00
66.00:297.00:33.00


In [119]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range $request_arg --async -f result_test.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 297 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 66 inference requests per second
  Client: 
    Request count: 1177
    Throughput: 63.4895 infer/sec
    Avg latency: 11440 usec (standard deviation 3672 usec)
    p50 latency: 9765 usec
    p90 latency: 16339 usec
    p95 latency: 18821 usec
    p99 latency: 25492 usec
    Avg HTTP time: 11412 usec (send/recv 1107 usec + response wait 10305 usec)
  Server: 
    Inference count: 1177
    Execution count: 1177
    Successful request count: 1177
    Avg request latency: 8211 usec (overhead 42 usec + queue 2161 usec + compute input 188 usec + compute infer 5792 usec + compute output 26 usec)
Request Rate: 99 inference requests per 

In [112]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 33:297:33 --async -f result_test.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 297 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 33 inference requests per second
Failed to obtain stable measurement.
Failed to obtain stable measurement.



: 2

In [118]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 66 --async -f result_test.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 66 inference requests per second
  Client: 
    Request count: 1181
    Throughput: 63.4872 infer/sec
    Avg latency: 11337 usec (standard deviation 3668 usec)
    p50 latency: 9671 usec
    p90 latency: 16294 usec
    p95 latency: 18700 usec
    p99 latency: 25029 usec
    Avg HTTP time: 11287 usec (send/recv 1117 usec + response wait 10170 usec)
  Server: 
    Inference count: 1181
    Execution count: 1181
    Successful request count: 1181
    Avg request latency: 8078 usec (overhead 42 usec + queue 2090 usec + compute input 187 usec + compute infer 5732 usec + compute output 26 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 66, throughput: 63.4872 infer/sec, latency 113

In [ ]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f result_test.csv

## Batching Experiment

Earlier, we noted that our model can achieve higher throughput with low latency by performing inference on batches of input samples, instead of individual samples. However, our client sends requests with individual samples.

To improve performance, we can ask the Triton server to batch incoming requests whenever possible, and send them through the server together instead of a sequence. In other words, if the server is ready to handle the next request, and it finds four requests waiting in the queue, it should serve those four as a batch instead of just taking the next request in line.

Let’s edit the model configuration:

``` bash
# runs on node-serve-system
nano ~/summer2025/models/food_classifier_onnx/config.pbtxt
```

and at the end, add

    dynamic_batching {
      preferred_batch_size: [4, 6, 8, 10]
      max_queue_delay_microseconds: 100
    }

Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

``` bash
# runs on node-serve-system
docker compose -f ~/exp-chi/docker/docker/docker-compose.yaml build triton_server
```

and then bring the server back up:

``` bash
# runs on node-serve-system
docker compose -f ~/exp-chi/docker/docker/docker-compose.yaml up triton_server --force-recreate -d
```

and use

``` bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

Before we benchmark this service again, let’s get some pre-benchmark stats about how many requests have been served, broken down by batch size. (If you’ve just restarted the server, it would be zero!)

In [154]:
# runs inside Jupyter container
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats 

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


Then let's run a benchmark. I will run benchmarks for a constant request rate of 100 (constant distribution) for various batch sizes {1,8,10,12,16} while the model's configuration has a set max batch number of 16.
I will then change the max batch number back to 1 and rerun a couple of benchmarks. It shouldn't allow me to run anything with a batch size greater than one.

Note: Reran first benchmarks after changing config again to save results in csv files.

In [155]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async -f batch1.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1905
    Throughput: 100.001 infer/sec
    Avg latency: 9454 usec (standard deviation 2274 usec)
    p50 latency: 9490 usec
    p90 latency: 10029 usec
    p95 latency: 10289 usec
    p99 latency: 12677 usec
    Avg HTTP time: 9341 usec (send/recv 491 usec + response wait 8850 usec)
  Server: 
    Inference count: 1905
    Execution count: 1905
    Successful request count: 1905
    Avg request latency: 6370 usec (overhead 40 usec + queue 137 usec + compute input 156 usec + compute infer 6011 usec + compute output 25 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 100.001 infer/sec, latency 9454 u

Get stats per batch

In [156]:
# runs inside Jupyter container
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1750104654444,"inference_count":3878,"execution_count":3878,"inference_stats":{"success":{"count":3878,"ns":2822039192172},"fail":{"count":0,"ns":0},"queue":{"count":3878,"ns":2794667220082},"compute_input":{"count":3878,"ns":575425899},"compute_infer":{"count":3878,"ns":26580321067},"compute_output":{"count":3878,"ns":84801843},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":3878,"ns":575425899},"compute_infer":{"count":3878,"ns":26580321067},"compute_output":{"count":3878,"ns":84801843}}],"memory_usage":[]}]}


Max Batch Number was set to 1 in the configuration. Now it is changed to 10 so let's run the benchmark again.

In [157]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 8 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async -f batch8.csv

*** Measurement Settings ***
  Batch size: 8
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1988
    Throughput: 800.134 infer/sec
    Avg latency: 13301 usec (standard deviation 4342 usec)
    p50 latency: 12626 usec
    p90 latency: 13868 usec
    p95 latency: 14726 usec
    p99 latency: 29348 usec
    Avg HTTP time: 13106 usec (send/recv 4005 usec + response wait 9101 usec)
  Server: 
    Inference count: 15904
    Execution count: 1988
    Successful request count: 1988
    Avg request latency: 6820 usec (overhead 22 usec + queue 200 usec + compute input 572 usec + compute infer 6010 usec + compute output 15 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 800.134 infer/sec, latency 1

In [158]:
# runs inside Jupyter container
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1750104778051,"inference_count":55494,"execution_count":10330,"inference_stats":{"success":{"count":10330,"ns":21019163113344},"fail":{"count":0,"ns":0},"queue":{"count":10330,"ns":20937878192070},"compute_input":{"count":10330,"ns":4276098761},"compute_infer":{"count":10330,"ns":76586698802},"compute_output":{"count":10330,"ns":168676035},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":3878,"ns":575425899},"compute_infer":{"count":3878,"ns":26580321067},"compute_output":{"count":3878,"ns":84801843}},{"batch_size":8,"compute_input":{"count":6452,"ns":3700672862},"compute_infer":{"count":6452,"ns":50006377735},"compute_output":{"count":6452,"ns":83874192}}],"memory_usage":[]}]}


In [159]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 16 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async -f batch16.csv

*** Measurement Settings ***
  Batch size: 16
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1573
    Throughput: 1315.81 infer/sec
    Avg latency: 11139137 usec (standard deviation 72627 usec)
    p50 latency: 6505055 usec
    p90 latency: 18334846 usec
    p95 latency: 18869078 usec
    p99 latency: 19328583 usec
    Avg HTTP time: 11148685 usec (send/recv 4189141 usec + response wait 6959544 usec)
  Server: 
    Inference count: 25168
    Execution count: 1573
    Successful request count: 1573
    Avg request latency: 5502057 usec (overhead 32 usec + queue 5489924 usec + compute input 1233 usec + compute infer 10840 usec + compute output 28 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throug

In [160]:
# runs inside Jupyter container
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1750104938614,"inference_count":159062,"execution_count":16803,"inference_stats":{"success":{"count":16803,"ns":46389981414662},"fail":{"count":0,"ns":0},"queue":{"count":16803,"ns":46227405651444},"compute_input":{"count":16803,"ns":13195332776},"compute_infer":{"count":16803,"ns":148603962452},"compute_output":{"count":16803,"ns":333853326},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":3878,"ns":575425899},"compute_infer":{"count":3878,"ns":26580321067},"compute_output":{"count":3878,"ns":84801843}},{"batch_size":8,"compute_input":{"count":6452,"ns":3700672862},"compute_infer":{"count":6452,"ns":50006377735},"compute_output":{"count":6452,"ns":83874192}},{"batch_size":16,"compute_input":{"count":6473,"ns":8919234015},"compute_infer":{"count":6473,"ns":72017263650},"compute_output":{"count":6473,"ns":165177291}}],"m

In [161]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 12 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async -f batch12.csv

*** Measurement Settings ***
  Batch size: 12
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 2001
    Throughput: 1266.01 infer/sec
    Avg latency: 11482101 usec (standard deviation 78780 usec)
    p50 latency: 15692173 usec
    p90 latency: 21291909 usec
    p95 latency: 21462857 usec
    p99 latency: 21558780 usec
    Avg HTTP time: 11474427 usec (send/recv 6894382 usec + response wait 4580045 usec)
  Server: 
    Inference count: 24036
    Execution count: 2003
    Successful request count: 2003
    Avg request latency: 977366 usec (overhead 29 usec + queue 967909 usec + compute input 1083 usec + compute infer 8323 usec + compute output 21 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughp

In [162]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 10 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async -f batch10.csv

*** Measurement Settings ***
  Batch size: 10
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1926
    Throughput: 1000.21 infer/sec
    Avg latency: 16407 usec (standard deviation 2974 usec)
    p50 latency: 15821 usec
    p90 latency: 16753 usec
    p95 latency: 19954 usec
    p99 latency: 32344 usec
    Avg HTTP time: 16350 usec (send/recv 4309 usec + response wait 12041 usec)
  Server: 
    Inference count: 19250
    Execution count: 1925
    Successful request count: 1925
    Avg request latency: 8688 usec (overhead 22 usec + queue 459 usec + compute input 822 usec + compute infer 7368 usec + compute output 15 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 1000.21 infer/sec, latency

In [167]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 14 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async -f batch14.csv

*** Measurement Settings ***
  Batch size: 14
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1778
    Throughput: 1313.12 infer/sec
    Avg latency: 4093885 usec (standard deviation 80487 usec)
    p50 latency: 4960687 usec
    p90 latency: 6151674 usec
    p95 latency: 6185497 usec
    p99 latency: 6221961 usec
    Avg HTTP time: 4096567 usec (send/recv 2113403 usec + response wait 1983164 usec)
  Server: 
    Inference count: 24906
    Execution count: 1779
    Successful request count: 1779
    Avg request latency: 1260688 usec (overhead 26 usec + queue 1250052 usec + compute input 1195 usec + compute infer 9394 usec + compute output 20 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 

In [166]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 14 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async -f batch14.csv

*** Measurement Settings ***
  Batch size: 14
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
Failed to obtain stable measurement.
Failed to obtain stable measurement.



: 2

In [164]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 4 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async -f batch4.csv

*** Measurement Settings ***
  Batch size: 4
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1958
    Throughput: 399.993 infer/sec
    Avg latency: 13504 usec (standard deviation 4333 usec)
    p50 latency: 13197 usec
    p90 latency: 14326 usec
    p95 latency: 15646 usec
    p99 latency: 25560 usec
    Avg HTTP time: 13333 usec (send/recv 2783 usec + response wait 10550 usec)
  Server: 
    Inference count: 7832
    Execution count: 1958
    Successful request count: 1958
    Avg request latency: 6607 usec (overhead 38 usec + queue 241 usec + compute input 495 usec + compute infer 5808 usec + compute output 24 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 399.993 infer/sec, latency 1

In [165]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 6 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async -f batch6.csv

*** Measurement Settings ***
  Batch size: 6
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1911
    Throughput: 600.047 infer/sec
    Avg latency: 13628 usec (standard deviation 2321 usec)
    p50 latency: 13657 usec
    p90 latency: 14445 usec
    p95 latency: 14651 usec
    p99 latency: 19401 usec
    Avg HTTP time: 13517 usec (send/recv 2699 usec + response wait 10818 usec)
  Server: 
    Inference count: 11466
    Execution count: 1911
    Successful request count: 1911
    Avg request latency: 6483 usec (overhead 39 usec + queue 123 usec + compute input 695 usec + compute infer 5600 usec + compute output 26 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 600.047 infer/sec, latency 

Changed max batch size to 1 again in config file to see what would happen without batches.

In [152]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1908
    Throughput: 99.9963 infer/sec
    Avg latency: 9849 usec (standard deviation 2974 usec)
    p50 latency: 9818 usec
    p90 latency: 10559 usec
    p95 latency: 11348 usec
    p99 latency: 12183 usec
    Avg HTTP time: 9688 usec (send/recv 597 usec + response wait 9091 usec)
  Server: 
    Inference count: 1908
    Execution count: 1908
    Successful request count: 1908
    Avg request latency: 6479 usec (overhead 42 usec + queue 143 usec + compute input 194 usec + compute infer 6073 usec + compute output 26 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 99.9963 infer/sec, latency 9849 u

In [153]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 8 --shape IMAGE:3,224,224 \
    --request-distribution=constant --request-rate-range 100 --async 

*** Measurement Settings ***
  Batch size: 8
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using uniform distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
Failed to retrieve results from inference request.
Thread [0] had error: [request id: 0] inference request batch-size must be <= 1 for 'food_classifier_onnx'

Thread [1] had error: [request id: 281474976710656] inference request batch-size must be <= 1 for 'food_classifier_onnx'

Thread [2] had error: [request id: 562949953421312] inference request batch-size must be <= 1 for 'food_classifier_onnx'

Thread [3] had error: [request id: 844424930131968] inference request batch-size must be <= 1 for 'food_classifier_onnx'




: 99

Tells us it is not possible to do batches of size 8 because the configuration only allows max batch size of 1. Not worth testing bigger batches - since it is not going to work.

## Dynamic Batching with varying λ

The goal of this experiment is to run the benchmark with:
 - Poisson arrival distribution (one by one arrivals)
 - varying λ values in the range {0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3} ms (must convert to usec)
 - dynamic batching with preferred batch sizes 1-8

 Then get the waiting time in queue for each λ and plot it against the theory (from jupyter notebook).

First we need to edit the model's configuration:

nano ~/summer2025/models/food_classifier_onnx/config.pbtxt

and at the end, add

dynamic_batching {

  preferred_batch_size: [1, 2, 4, 6, 8]
  
  max_queue_delay_microseconds: 100
  
}
Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

docker compose -f ~/summer2025/docker/docker-compose.yaml build triton_server
and then bring the server back up:

docker compose -f ~/summer2025/docker/docker-compose.yaml up triton_server --force-recreate -d

Note! 07/22/2025 reran this experiment (using compute gigaio gpu)

In [75]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --async -f batch_test_500.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 10013
    Avg send request rate: 506.34 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.05% of the requests were delayed. 
    Throughput: 503.20 infer/sec
    Avg latency: 5927 usec (standard deviation 3486 usec)
    p50 latency: 5307 usec
    p90 latency: 8488 usec
    p95 latency: 10041 usec
    p99 latency: 15626 usec
    Avg HTTP time: 5845 usec (send/recv 370 usec + response wait 5475 usec)
  Server: 
    Inference count: 10013
    Execution count: 6686
    Successful request count: 10013
    Avg request latency: 3449 usec (overhead 23 usec + queue 998 usec + compute input 608 usec + 

In [76]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753192436219,"inference_count":35638,"execution_count":18921,"inference_stats":{"success":{"count":35638,"ns":8220901541680},"fail":{"count":0,"ns":0},"queue":{"count":35638,"ns":8075188303261},"compute_input":{"count":35638,"ns":39147600233},"compute_infer":{"count":35638,"ns":104822562113},"compute_output":{"count":35638,"ns":489256893},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":11193,"ns":3213329561},"compute_infer":{"count":11193,"ns":24228428664},"compute_output":{"count":11193,"ns":98240664}},{"batch_size":2,"compute_input":{"count":4132,"ns":3835156064},"compute_infer":{"count":4132,"ns":8339264764},"compute_output":{"count":4132,"ns":45890587}},{"batch_size":3,"compute_input":{"count":2122,"ns":3164210143},"compute_infer":{"count":2122,"ns":4764379129},"compute_output":{"count":2122,"ns":28295548}},{"batc

In [77]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 700 --async -f batch_test_700.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 700 inference requests per second
  Client: 
    Request count: 18267
    Avg send request rate: 686.66 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.88% of the requests were delayed. 
    Throughput: 712.09 infer/sec
    Avg latency: 8297 usec (standard deviation 12261 usec)
    p50 latency: 6087 usec
    p90 latency: 10817 usec
    p95 latency: 19262 usec
    p99 latency: 52234 usec
    Avg HTTP time: 8099 usec (send/recv 785 usec + response wait 7314 usec)
  Server: 
    Inference count: 18268
    Execution count: 9253
    Successful request count: 18268
    Avg request latency: 4036 usec (overhead 31 usec + queue 1300 usec + compute input 811 usec

In [78]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753192497882,"inference_count":59228,"execution_count":30974,"inference_stats":{"success":{"count":59228,"ns":8315808936099},"fail":{"count":0,"ns":0},"queue":{"count":59228,"ns":8106139513726},"compute_input":{"count":59228,"ns":57662005496},"compute_infer":{"count":59228,"ns":149303252376},"compute_output":{"count":59228,"ns":756752060},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":16771,"ns":4525846766},"compute_infer":{"count":16771,"ns":34796068288},"compute_output":{"count":16771,"ns":145962609}},{"batch_size":2,"compute_input":{"count":7541,"ns":5839191962},"compute_infer":{"count":7541,"ns":14580816811},"compute_output":{"count":7541,"ns":80526573}},{"batch_size":3,"compute_input":{"count":4214,"ns":5018967826},"compute_infer":{"count":4214,"ns":8638832631},"compute_output":{"count":4214,"ns":52460319}},{"ba

In [79]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 900 --async -f batch_test_900.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 900 inference requests per second
  Client: 
    Request count: 21578
    Avg send request rate: 913.99 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.87% of the requests were delayed. 
    Throughput: 905.70 infer/sec
    Avg latency: 10736 usec (standard deviation 19907 usec)
    p50 latency: 6480 usec
    p90 latency: 12754 usec
    p95 latency: 30713 usec
    p99 latency: 103263 usec
    Avg HTTP time: 10574 usec (send/recv 1182 usec + response wait 9392 usec)
  Server: 
    Inference count: 21579
    Execution count: 9954
    Successful request count: 21579
    Avg request latency: 3921 usec (overhead 34 usec + queue 1557 usec + compute input 508 

In [80]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 1100 --async -f batch_test_1100.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 1100 inference requests per second
  Client: 
    Request count: 31567
    Avg send request rate: 1078.47 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 5.17% of the requests were delayed. 
    Throughput: 1099.81 infer/sec
    Avg latency: 12257 usec (standard deviation 23873 usec)
    p50 latency: 6012 usec
    p90 latency: 14791 usec
    p95 latency: 58312 usec
    p99 latency: 128679 usec
    Avg HTTP time: 12108 usec (send/recv 1452 usec + response wait 10656 usec)
  Server: 
    Inference count: 31565
    Execution count: 12627
    Successful request count: 31565
    Avg request latency: 4401 usec (overhead 38 usec + queue 2059 usec + compute input

In [81]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 600 --async -f batch_test_600.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 600 inference requests per second
  Client: 
    Request count: 12924
    Avg send request rate: 587.05 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.23% of the requests were delayed. 
    Throughput: 598.67 infer/sec
    Avg latency: 5727 usec (standard deviation 7347 usec)
    p50 latency: 4746 usec
    p90 latency: 7327 usec
    p95 latency: 8842 usec
    p99 latency: 29323 usec
    Avg HTTP time: 5616 usec (send/recv 631 usec + response wait 4985 usec)
  Server: 
    Inference count: 12924
    Execution count: 8793
    Successful request count: 12924
    Avg request latency: 2694 usec (overhead 20 usec + queue 806 usec + compute input 169 usec + c

In [82]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --async -f batch_test_800.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 800 inference requests per second
  Client: 
    Request count: 19228
    Avg send request rate: 826.37 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.18% of the requests were delayed. 
    Throughput: 813.26 infer/sec
    Avg latency: 7405 usec (standard deviation 12911 usec)
    p50 latency: 5024 usec
    p90 latency: 8921 usec
    p95 latency: 17068 usec
    p99 latency: 51916 usec
    Avg HTTP time: 7270 usec (send/recv 837 usec + response wait 6433 usec)
  Server: 
    Inference count: 19229
    Execution count: 10596
    Successful request count: 19229
    Avg request latency: 3079 usec (overhead 24 usec + queue 1048 usec + compute input 255 usec

In [83]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 1000 --async -f batch_test_1000.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 1000 inference requests per second
  Client: 
    Request count: 23947
    Avg send request rate: 936.94 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 4.76% of the requests were delayed. 
    Throughput: 1000.76 infer/sec
    Avg latency: 9887 usec (standard deviation 20698 usec)
    p50 latency: 5392 usec
    p90 latency: 9822 usec
    p95 latency: 35849 usec
    p99 latency: 106651 usec
    Avg HTTP time: 9693 usec (send/recv 1291 usec + response wait 8402 usec)
  Server: 
    Inference count: 23949
    Execution count: 10729
    Successful request count: 23949
    Avg request latency: 3759 usec (overhead 32 usec + queue 1530 usec + compute input 372 

In [84]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 1200 --async -f batch_test_1200.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 1200 inference requests per second
  Client: 
    Request count: 31165
    Avg send request rate: 1166.12 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 4.13% of the requests were delayed. 
    Throughput: 1194.97 infer/sec
    Avg latency: 10920 usec (standard deviation 18402 usec)
    p50 latency: 6317 usec
    p90 latency: 14240 usec
    p95 latency: 37577 usec
    p99 latency: 100218 usec
    Avg HTTP time: 10790 usec (send/recv 1232 usec + response wait 9558 usec)
  Server: 
    Inference count: 31165
    Execution count: 11313
    Successful request count: 31163
    Avg request latency: 4127 usec (overhead 35 usec + queue 1724 usec + compute input 

## Dynamic Batching with varying λ run 2

In [86]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --async -f batch_test2_500.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 10626
    Avg send request rate: 514.30 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.73% of the requests were delayed. 
    Throughput: 511.66 infer/sec
    Avg latency: 5991 usec (standard deviation 4745 usec)
    p50 latency: 5124 usec
    p90 latency: 8535 usec
    p95 latency: 10349 usec
    p99 latency: 24732 usec
    Avg HTTP time: 5918 usec (send/recv 543 usec + response wait 5375 usec)
  Server: 
    Inference count: 10627
    Execution count: 7649
    Successful request count: 10627
    Avg request latency: 2806 usec (overhead 21 usec + queue 797 usec + compute input 295 usec + 

In [87]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 600 --async -f batch_test2_600.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 600 inference requests per second
  Client: 
    Request count: 13669
    Avg send request rate: 620.30 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.36% of the requests were delayed. 
    Throughput: 605.29 infer/sec
    Avg latency: 9469 usec (standard deviation 8332 usec)
    p50 latency: 7845 usec
    p90 latency: 15122 usec
    p95 latency: 18942 usec
    p99 latency: 40874 usec
    Avg HTTP time: 9349 usec (send/recv 644 usec + response wait 8705 usec)
  Server: 
    Inference count: 13669
    Execution count: 7196
    Successful request count: 13669
    Avg request latency: 4682 usec (overhead 32 usec + queue 1480 usec + compute input 1308 usec

In [88]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 700 --async -f batch_test2_700.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 700 inference requests per second
  Client: 
    Request count: 14794
    Avg send request rate: 734.01 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.14% of the requests were delayed. 
    Throughput: 704.74 infer/sec
    Avg latency: 6632 usec (standard deviation 6826 usec)
    p50 latency: 5612 usec
    p90 latency: 9176 usec
    p95 latency: 11393 usec
    p99 latency: 28043 usec
    Avg HTTP time: 6508 usec (send/recv 516 usec + response wait 5992 usec)
  Server: 
    Inference count: 14794
    Execution count: 7936
    Successful request count: 14794
    Avg request latency: 3718 usec (overhead 27 usec + queue 1169 usec + compute input 669 usec +

In [89]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --async -f batch_test2_800.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 800 inference requests per second
  Client: 
    Request count: 32120
    Avg send request rate: 671.83 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 7.96% of the requests were delayed. 
    Throughput: 800.88 infer/sec
    Avg latency: 18946 usec (standard deviation 16261 usec)
    p50 latency: 5985 usec
    p90 latency: 34252 usec
    p95 latency: 115135 usec
    p99 latency: 236693 usec
    Avg HTTP time: 18557 usec (send/recv 3486 usec + response wait 15071 usec)
  Server: 
    Inference count: 32121
    Execution count: 15038
    Successful request count: 32121
    Avg request latency: 5358 usec (overhead 31 usec + queue 2703 usec + compute input 7

In [90]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 900 --async -f batch_test2_900.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 900 inference requests per second
  Client: 
    Request count: 19995
    Avg send request rate: 1066.28 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.76% of the requests were delayed. 
    Throughput: 915.69 infer/sec
    Avg latency: 8053 usec (standard deviation 11809 usec)
    p50 latency: 6106 usec
    p90 latency: 10979 usec
    p95 latency: 15037 usec
    p99 latency: 45592 usec
    Avg HTTP time: 7922 usec (send/recv 552 usec + response wait 7370 usec)
  Server: 
    Inference count: 19990
    Execution count: 8130
    Successful request count: 19990
    Avg request latency: 4520 usec (overhead 34 usec + queue 1658 usec + compute input 922 use

In [91]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 1000 --async -f batch_test2_1000.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 1000 inference requests per second
  Client: 
    Request count: 23121
    Avg send request rate: 1016.91 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.36% of the requests were delayed. 
    Throughput: 1000.01 infer/sec
    Avg latency: 8675 usec (standard deviation 12570 usec)
    p50 latency: 6267 usec
    p90 latency: 11211 usec
    p95 latency: 15970 usec
    p99 latency: 71955 usec
    Avg HTTP time: 8538 usec (send/recv 805 usec + response wait 7733 usec)
  Server: 
    Inference count: 23120
    Execution count: 9130
    Successful request count: 23120
    Avg request latency: 4184 usec (overhead 35 usec + queue 1507 usec + compute input 739 u

In [92]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 1100 --async -f batch_test2_1100.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 1100 inference requests per second
  Client: 
    Request count: 25639
    Avg send request rate: 1074.59 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.82% of the requests were delayed. 
    Throughput: 1096.90 infer/sec
    Avg latency: 13254 usec (standard deviation 17819 usec)
    p50 latency: 7643 usec
    p90 latency: 26860 usec
    p95 latency: 38122 usec
    p99 latency: 84251 usec
    Avg HTTP time: 13142 usec (send/recv 780 usec + response wait 12362 usec)
  Server: 
    Inference count: 25637
    Execution count: 9449
    Successful request count: 25637
    Avg request latency: 4885 usec (overhead 44 usec + queue 2197 usec + compute input 80

In [93]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 1200 --async -f batch_test2_1200.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 1200 inference requests per second
  Client: 
    Request count: 30650
    Avg send request rate: 1261.98 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.99% of the requests were delayed. 
    Throughput: 1203.04 infer/sec
    Avg latency: 13700 usec (standard deviation 18209 usec)
    p50 latency: 8041 usec
    p90 latency: 24718 usec
    p95 latency: 44384 usec
    p99 latency: 98699 usec
    Avg HTTP time: 13566 usec (send/recv 1122 usec + response wait 12444 usec)
  Server: 
    Inference count: 30651
    Execution count: 10461
    Successful request count: 30651
    Avg request latency: 4315 usec (overhead 34 usec + queue 1726 usec + compute input 

In [94]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 1300 --async -f batch_test2_1300.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 1300 inference requests per second
  Client: 
    Request count: 38613
    Avg send request rate: 1326.32 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 4.94% of the requests were delayed. 
    Throughput: 1313.60 infer/sec
    Avg latency: 11968 usec (standard deviation 15982 usec)
    p50 latency: 5587 usec
    p90 latency: 12527 usec
    p95 latency: 51402 usec
    p99 latency: 132912 usec
    Avg HTTP time: 11814 usec (send/recv 1661 usec + response wait 10153 usec)
  Server: 
    Inference count: 38612
    Execution count: 13369
    Successful request count: 38612
    Avg request latency: 4339 usec (overhead 36 usec + queue 2024 usec + compute input

## Focusing on the 0.8 arrival rate point

In [16]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [17]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --async -f batch_new_test_800.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 800 inference requests per second
  Client: 
    Request count: 19275
    Avg send request rate: 832.47 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.17% of the requests were delayed. 
    Throughput: 811.63 infer/sec
    Avg latency: 6933 usec (standard deviation 9198 usec)
    p50 latency: 5398 usec
    p90 latency: 9181 usec
    p95 latency: 11488 usec
    p99 latency: 52272 usec
    Avg HTTP time: 6812 usec (send/recv 494 usec + response wait 6318 usec)
  Server: 
    Inference count: 19276
    Execution count: 10133
    Successful request count: 19276
    Avg request latency: 3096 usec (overhead 27 usec + queue 984 usec + compute input 252 usec +

In [13]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1752520379863,"inference_count":65676,"execution_count":26143,"inference_stats":{"success":{"count":65676,"ns":16398167533574},"fail":{"count":0,"ns":0},"queue":{"count":65676,"ns":16170544788576},"compute_input":{"count":65676,"ns":63671058357},"compute_infer":{"count":65676,"ns":160340058509},"compute_output":{"count":65676,"ns":965886482},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":11387,"ns":1680892048},"compute_infer":{"count":11387,"ns":23615000207},"compute_output":{"count":11387,"ns":103180849}},{"batch_size":2,"compute_input":{"count":5689,"ns":3225072651},"compute_infer":{"count":5689,"ns":10677292419},"compute_output":{"count":5689,"ns":63468000}},{"batch_size":3,"compute_input":{"count":3497,"ns":3204112234},"compute_infer":{"count":3497,"ns":6911102658},"compute_output":{"count":3497,"ns":46424694}},{"

Conclusion: 

After plotting the average compute inference time per batch we realized that it is not constant, which contradicts our assumption.

Without a constant service time the rest of our simplifications fall apart. 

The next step is to figure out why compute time is not constant.

## Figuring out what is going wrong

First we need to capture the requests for one run of the benchmark using tcdump: 

sudo apt install tcpdump

ip address (to figure out where docker runs from)

sudo tcpdump -w requests.pcap -i br-4b199d5448bb tcp port 8000 (the triton server should be running at the same time so that there are requests to capture)


Then after the requests are captured in the csv file requests.csv, we need to copy the csv file onto my local directory. Exit the ssh and from sof run this:

scp -i ~/.ssh/id_rsa_chameleon cc@192.5.87.156:~/requests.pcap ~/Desktop/
 
To get info about the requests captured we need to use the terminal extension of wireshark (tshark) to get the request and response timestamps and save those in csv files:

tshark -r requests.pcap -Y "http.request" -T fields -e frame.time_relative > timestamps_request.csv

tshark -r requests.pcap -Y "http.request" -T fields -e frame.time_relative > timestamps_response.csv

Notes from running those: 

1467867 packets captured

1639255 packets received by filter

171388 packets dropped by kernel

## Generating pcap for 400 req per sec

In [19]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [21]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async -f batch_new_test_400.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 8513
    Avg send request rate: 407.95 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.02% of the requests were delayed. 
    Throughput: 407.99 infer/sec
    Avg latency: 5039 usec (standard deviation 3489 usec)
    p50 latency: 4531 usec
    p90 latency: 6869 usec
    p95 latency: 7795 usec
    p99 latency: 11944 usec
    Avg HTTP time: 4953 usec (send/recv 480 usec + response wait 4473 usec)
  Server: 
    Inference count: 8513
    Execution count: 6583
    Successful request count: 8513
    Avg request latency: 2655 usec (overhead 19 usec + queue 669 usec + compute input 130 usec + comp

In [22]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


## Changing the dynamic batching config to observe changes in performance

In [23]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --async -f batch_new_config_800.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 800 inference requests per second
  Client: 
    Request count: 19013
    Avg send request rate: 789.90 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.54% of the requests were delayed. 
    Throughput: 812.63 infer/sec
    Avg latency: 9732 usec (standard deviation 11429 usec)
    p50 latency: 6939 usec
    p90 latency: 15093 usec
    p95 latency: 22724 usec
    p99 latency: 62601 usec
    Avg HTTP time: 9602 usec (send/recv 598 usec + response wait 9004 usec)
  Server: 
    Inference count: 19012
    Execution count: 8777
    Successful request count: 19012
    Avg request latency: 3906 usec (overhead 32 usec + queue 1225 usec + compute input 793 usec

In [27]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [25]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async -f batch_new_config_400.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 8819
    Avg send request rate: 419.39 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 2.06% of the requests were delayed. 
    Throughput: 413.38 infer/sec
    Avg latency: 5958 usec (standard deviation 5030 usec)
    p50 latency: 5264 usec
    p90 latency: 8022 usec
    p95 latency: 9517 usec
    p99 latency: 18439 usec
    Avg HTTP time: 5844 usec (send/recv 615 usec + response wait 5229 usec)
  Server: 
    Inference count: 8819
    Execution count: 6339
    Successful request count: 8819
    Avg request latency: 3365 usec (overhead 21 usec + queue 860 usec + compute input 554 usec + comp

In [26]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1752700758323,"inference_count":82623,"execution_count":37475,"inference_stats":{"success":{"count":82623,"ns":18471792767292},"fail":{"count":0,"ns":0},"queue":{"count":82623,"ns":18199754767871},"compute_input":{"count":82623,"ns":69801195028},"compute_infer":{"count":82623,"ns":197995029778},"compute_output":{"count":82623,"ns":1190070360},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":19248,"ns":4072310950},"compute_infer":{"count":19248,"ns":40533030902},"compute_output":{"count":19248,"ns":167286673}},{"batch_size":2,"compute_input":{"count":7872,"ns":4258994850},"compute_infer":{"count":7872,"ns":15027462777},"compute_output":{"count":7872,"ns":82244486}},{"batch_size":3,"compute_input":{"count":4371,"ns":3607081623},"compute_infer":{"count":4371,"ns":8796998745},"compute_output":{"count":4371,"ns":53581288}},{

In [28]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async -f batch_new_config_400.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 8153
    Avg send request rate: 391.80 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.08% of the requests were delayed. 
    Throughput: 400.43 infer/sec
    Avg latency: 37220 usec (standard deviation 44798 usec)
    p50 latency: 5036 usec
    p90 latency: 8743 usec
    p95 latency: 159334 usec
    p99 latency: 917661 usec
    Avg HTTP time: 37123 usec (send/recv 430 usec + response wait 36693 usec)
  Server: 
    Inference count: 8151
    Execution count: 5735
    Successful request count: 8151
    Avg request latency: 34767 usec (overhead 32 usec + queue 30375 usec + compute input 663 u

In [29]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1752701104209,"inference_count":28733,"execution_count":13899,"inference_stats":{"success":{"count":28733,"ns":9452307608236},"fail":{"count":0,"ns":0},"queue":{"count":28733,"ns":8997976881043},"compute_input":{"count":28733,"ns":72810767546},"compute_infer":{"count":28733,"ns":377668118671},"compute_output":{"count":28733,"ns":942602185},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":10200,"ns":2322701020},"compute_infer":{"count":10200,"ns":22818035704},"compute_output":{"count":10200,"ns":85404885}},{"batch_size":2,"compute_input":{"count":2298,"ns":1443412247},"compute_infer":{"count":2298,"ns":5205614922},"compute_output":{"count":2298,"ns":22554938}},{"batch_size":3,"compute_input":{"count":764,"ns":760244447},"compute_infer":{"count":764,"ns":2389860437},"compute_output":{"count":764,"ns":8555878}},{"batch_siz

Now config is only

dynamic batching {

    max queue delay in microseconds = 100
}

In [30]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [31]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async -f batch_new_config_400.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 7589
    Throughput: 397.257 infer/sec
    Avg latency: 5312 usec (standard deviation 2618 usec)
    p50 latency: 4825 usec
    p90 latency: 7586 usec
    p95 latency: 8651 usec
    p99 latency: 11294 usec
    Avg HTTP time: 5230 usec (send/recv 375 usec + response wait 4855 usec)
  Server: 
    Inference count: 7587
    Execution count: 6165
    Successful request count: 7587
    Avg request latency: 3173 usec (overhead 17 usec + queue 1076 usec + compute input 266 usec + compute infer 1804 usec + compute output 8 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 400, throughput: 397.257 infer/sec, latency 5312 use

In [32]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1752777914560,"inference_count":27228,"execution_count":13548,"inference_stats":{"success":{"count":27228,"ns":9965576408105},"fail":{"count":0,"ns":0},"queue":{"count":27228,"ns":9477247114701},"compute_input":{"count":27228,"ns":58060624964},"compute_infer":{"count":27228,"ns":426668246175},"compute_output":{"count":27228,"ns":881853238},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":10687,"ns":634669749},"compute_infer":{"count":10687,"ns":22568359397},"compute_output":{"count":10687,"ns":88810495}},{"batch_size":2,"compute_input":{"count":2054,"ns":1192569974},"compute_infer":{"count":2054,"ns":4576926297},"compute_output":{"count":2054,"ns":19497228}},{"batch_size":3,"compute_input":{"count":344,"ns":290920592},"compute_infer":{"count":344,"ns":1497010486},"compute_output":{"count":344,"ns":3653133}},{"batch_size

just changed max batch size to 64 to see the difference

In [33]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [34]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async -f batch_new_config_400.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 7614
    Throughput: 396.502 infer/sec
    Avg latency: 5222 usec (standard deviation 2447 usec)
    p50 latency: 4694 usec
    p90 latency: 7451 usec
    p95 latency: 8464 usec
    p99 latency: 11080 usec
    Avg HTTP time: 5179 usec (send/recv 371 usec + response wait 4808 usec)
  Server: 
    Inference count: 7614
    Execution count: 6342
    Successful request count: 7614
    Avg request latency: 2749 usec (overhead 18 usec + queue 915 usec + compute input 116 usec + compute infer 1691 usec + compute output 9 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 400, throughput: 396.502 infer/sec, latency 5222 usec

In [35]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1752778339459,"inference_count":27053,"execution_count":8257,"inference_stats":{"success":{"count":27053,"ns":17231648855778},"fail":{"count":0,"ns":0},"queue":{"count":27053,"ns":15762225734606},"compute_input":{"count":27053,"ns":211964038356},"compute_infer":{"count":27053,"ns":1244678289940},"compute_output":{"count":27053,"ns":3068408468},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":6599,"ns":541718794},"compute_infer":{"count":6599,"ns":14487226052},"compute_output":{"count":6599,"ns":57454221}},{"batch_size":2,"compute_input":{"count":1123,"ns":306788082},"compute_infer":{"count":1123,"ns":2912599696},"compute_output":{"count":1123,"ns":11712545}},{"batch_size":3,"compute_input":{"count":189,"ns":91561301},"compute_infer":{"count":189,"ns":1280600669},"compute_output":{"count":189,"ns":2337190}},{"batch_size"

now added preferred batch size = [64] in dynamic batching config

In [48]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [49]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async -f batch_new_config_400.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 7860
    Avg send request rate: 389.42 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.40% of the requests were delayed. 
    Throughput: 400.18 infer/sec
    Avg latency: 121887 usec (standard deviation 30826 usec)
    p50 latency: 6449 usec
    p90 latency: 575885 usec
    p95 latency: 843728 usec
    p99 latency: 1192676 usec
    Avg HTTP time: 121786 usec (send/recv 553 usec + response wait 121233 usec)
  Server: 
    Inference count: 7863
    Execution count: 5061
    Successful request count: 7863
    Avg request latency: 118566 usec (overhead 84 usec + queue 104189 usec + compute inp

In [50]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1752861305182,"inference_count":28317,"execution_count":5663,"inference_stats":{"success":{"count":28317,"ns":19975049060535},"fail":{"count":0,"ns":0},"queue":{"count":28317,"ns":18105283612900},"compute_input":{"count":28317,"ns":266919171598},"compute_infer":{"count":28317,"ns":1591277427035},"compute_output":{"count":28317,"ns":2506119665},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":4184,"ns":975918137},"compute_infer":{"count":4184,"ns":11430506797},"compute_output":{"count":4184,"ns":37115351}},{"batch_size":2,"compute_input":{"count":855,"ns":510165689},"compute_infer":{"count":855,"ns":2589213908},"compute_output":{"count":855,"ns":8674727}},{"batch_size":3,"compute_input":{"count":195,"ns":173422303},"compute_infer":{"count":195,"ns":1334932239},"compute_output":{"count":195,"ns":2178646}},{"batch_size":4,

too much queueuing - nothing really changed in terms of service time - back to testing without preferred batch size

In [51]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [52]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1956
    Throughput: 103.413 infer/sec
    Avg latency: 7685 usec (standard deviation 2158 usec)
    p50 latency: 6853 usec
    p90 latency: 10711 usec
    p95 latency: 11890 usec
    p99 latency: 14652 usec
    Avg HTTP time: 7641 usec (send/recv 434 usec + response wait 7207 usec)
  Server: 
    Inference count: 1956
    Execution count: 1841
    Successful request count: 1956
    Avg request latency: 5145 usec (overhead 34 usec + queue 1168 usec + compute input 348 usec + compute infer 3576 usec + compute output 18 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 103.413 infer/sec, latency 7685 

In [53]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1752861600037,"inference_count":5001,"execution_count":2276,"inference_stats":{"success":{"count":5001,"ns":4537092944084},"fail":{"count":0,"ns":0},"queue":{"count":5001,"ns":3592843563305},"compute_input":{"count":5001,"ns":33373262228},"compute_infer":{"count":5001,"ns":909136449439},"compute_output":{"count":5001,"ns":457061710},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2095,"ns":624090595},"compute_infer":{"count":2095,"ns":10826678477},"compute_output":{"count":2095,"ns":37621917}},{"batch_size":2,"compute_input":{"count":122,"ns":90435704},"compute_infer":{"count":122,"ns":1310770734},"compute_output":{"count":122,"ns":2531502}},{"batch_size":3,"compute_input":{"count":7,"ns":6984250},"compute_infer":{"count":7,"ns":912351152},"compute_output":{"count":7,"ns":196611}},{"batch_size":4,"compute_input":{"coun

Configuration: max batch size = 8, simplest dynamic batching, req rate = 400

In [55]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [56]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 7904
    Throughput: 399.793 infer/sec
    Avg latency: 5750 usec (standard deviation 3256 usec)
    p50 latency: 5143 usec
    p90 latency: 8438 usec
    p95 latency: 9895 usec
    p99 latency: 13599 usec
    Avg HTTP time: 5675 usec (send/recv 389 usec + response wait 5286 usec)
  Server: 
    Inference count: 7904
    Execution count: 5911
    Successful request count: 7904
    Avg request latency: 3128 usec (overhead 22 usec + queue 800 usec + compute input 461 usec + compute infer 1834 usec + compute output 10 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 400, throughput: 399.793 infer/sec, latency 5750 use

In [57]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753101653603,"inference_count":25529,"execution_count":15480,"inference_stats":{"success":{"count":25529,"ns":6397131516214},"fail":{"count":0,"ns":0},"queue":{"count":25529,"ns":6287168063828},"compute_input":{"count":25529,"ns":22392738052},"compute_infer":{"count":25529,"ns":86397484447},"compute_output":{"count":25529,"ns":330557782},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":11063,"ns":2909700571},"compute_infer":{"count":11063,"ns":24220537799},"compute_output":{"count":11063,"ns":97907596}},{"batch_size":2,"compute_input":{"count":2524,"ns":1926158543},"compute_infer":{"count":2524,"ns":5508234582},"compute_output":{"count":2524,"ns":28235461}},{"batch_size":3,"compute_input":{"count":887,"ns":1024240848},"compute_infer":{"count":887,"ns":2560747651},"compute_output":{"count":887,"ns":11601952}},{"batch_si

Configuration: max batch size = 64, simplest dynamic batching, req rate = 400

In [59]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [61]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 8516
    Avg send request rate: 396.10 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.15% of the requests were delayed. 
    Throughput: 407.94 infer/sec
    Avg latency: 4972 usec (standard deviation 2363 usec)
    p50 latency: 4545 usec
    p90 latency: 6929 usec
    p95 latency: 7924 usec
    p99 latency: 11232 usec
    Avg HTTP time: 4920 usec (send/recv 464 usec + response wait 4456 usec)
  Server: 
    Inference count: 8517
    Execution count: 6659
    Successful request count: 8517
    Avg request latency: 2611 usec (overhead 19 usec + queue 667 usec + compute input 137 usec + comp

In [62]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753101970066,"inference_count":38273,"execution_count":14876,"inference_stats":{"success":{"count":38273,"ns":15551106243236},"fail":{"count":0,"ns":0},"queue":{"count":38273,"ns":13748494832133},"compute_input":{"count":38273,"ns":149530754686},"compute_infer":{"count":38273,"ns":1641528180727},"compute_output":{"count":38273,"ns":2552507915},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":11793,"ns":735325204},"compute_infer":{"count":11793,"ns":23769712287},"compute_output":{"count":11793,"ns":99795721}},{"batch_size":2,"compute_input":{"count":1943,"ns":389966201},"compute_infer":{"count":1943,"ns":4202461667},"compute_output":{"count":1943,"ns":19526175}},{"batch_size":3,"compute_input":{"count":623,"ns":182102747},"compute_infer":{"count":623,"ns":1924485307},"compute_output":{"count":623,"ns":7028028}},{"batch_

Configuration: max batch size = 64, delayed dynamic batching, req rate = 400

In [63]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [64]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 7616
    Throughput: 396.975 infer/sec
    Avg latency: 5467 usec (standard deviation 2366 usec)
    p50 latency: 5035 usec
    p90 latency: 7694 usec
    p95 latency: 8710 usec
    p99 latency: 11496 usec
    Avg HTTP time: 5402 usec (send/recv 361 usec + response wait 5041 usec)
  Server: 
    Inference count: 7616
    Execution count: 6212
    Successful request count: 7616
    Avg request latency: 3299 usec (overhead 19 usec + queue 1148 usec + compute input 262 usec + compute infer 1861 usec + compute output 8 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 400, throughput: 396.975 infer/sec, latency 5467 use

In [65]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753102226719,"inference_count":27454,"execution_count":8214,"inference_stats":{"success":{"count":27454,"ns":15035622108466},"fail":{"count":0,"ns":0},"queue":{"count":27454,"ns":13574724648588},"compute_input":{"count":27454,"ns":180608659167},"compute_infer":{"count":27454,"ns":1271054500349},"compute_output":{"count":27454,"ns":1917295911},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":6436,"ns":373756371},"compute_infer":{"count":6436,"ns":15330898544},"compute_output":{"count":6436,"ns":52020824}},{"batch_size":2,"compute_input":{"count":1195,"ns":702553241},"compute_infer":{"count":1195,"ns":3049692826},"compute_output":{"count":1195,"ns":11201753}},{"batch_size":3,"compute_input":{"count":229,"ns":193589133},"compute_infer":{"count":229,"ns":1263245210},"compute_output":{"count":229,"ns":2657821}},{"batch_size

Configuration: max batch size = 8, delayed dynamic batching, req rate = 400

In [66]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [67]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 8015
    Avg send request rate: 383.41 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.24% of the requests were delayed. 
    Throughput: 396.70 infer/sec
    Avg latency: 6154 usec (standard deviation 4292 usec)
    p50 latency: 5472 usec
    p90 latency: 8857 usec
    p95 latency: 10332 usec
    p99 latency: 15502 usec
    Avg HTTP time: 6040 usec (send/recv 416 usec + response wait 5624 usec)
  Server: 
    Inference count: 8015
    Execution count: 6349
    Successful request count: 8015
    Avg request latency: 3568 usec (overhead 20 usec + queue 1289 usec + compute input 401 usec + co

In [68]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753102463136,"inference_count":23195,"execution_count":14913,"inference_stats":{"success":{"count":23195,"ns":6480653526349},"fail":{"count":0,"ns":0},"queue":{"count":23195,"ns":6380462786216},"compute_input":{"count":23195,"ns":16503262880},"compute_infer":{"count":23195,"ns":82708638125},"compute_output":{"count":23195,"ns":294359546},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":11357,"ns":2575792127},"compute_infer":{"count":11357,"ns":24364877762},"compute_output":{"count":11357,"ns":96396152}},{"batch_size":2,"compute_input":{"count":2260,"ns":1276007204},"compute_infer":{"count":2260,"ns":5130665079},"compute_output":{"count":2260,"ns":22168834}},{"batch_size":3,"compute_input":{"count":489,"ns":408932423},"compute_infer":{"count":489,"ns":1861106388},"compute_output":{"count":489,"ns":5628924}},{"batch_size

Configuration: max batch size = 8, delayed dynamic batching with pref batch size [4,6,8], req rate = 400

In [69]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [70]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 8993
    Avg send request rate: 385.94 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.68% of the requests were delayed. 
    Throughput: 412.33 infer/sec
    Avg latency: 6795 usec (standard deviation 4430 usec)
    p50 latency: 5974 usec
    p90 latency: 9857 usec
    p95 latency: 11539 usec
    p99 latency: 22375 usec
    Avg HTTP time: 6723 usec (send/recv 413 usec + response wait 6310 usec)
  Server: 
    Inference count: 8994
    Execution count: 6824
    Successful request count: 8994
    Avg request latency: 4104 usec (overhead 25 usec + queue 1463 usec + compute input 767 usec + co

In [71]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753102706119,"inference_count":14407,"execution_count":7754,"inference_stats":{"success":{"count":14407,"ns":8272449789556},"fail":{"count":0,"ns":0},"queue":{"count":14407,"ns":8190903830798},"compute_input":{"count":14407,"ns":15441637815},"compute_infer":{"count":14407,"ns":65315205894},"compute_output":{"count":14407,"ns":218254103},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":5503,"ns":1741983491},"compute_infer":{"count":5503,"ns":13439470026},"compute_output":{"count":5503,"ns":51322443}},{"batch_size":2,"compute_input":{"count":1241,"ns":1272833099},"compute_infer":{"count":1241,"ns":3201860777},"compute_output":{"count":1241,"ns":14469148}},{"batch_size":3,"compute_input":{"count":245,"ns":380080245},"compute_infer":{"count":245,"ns":1404727088},"compute_output":{"count":245,"ns":3409214}},{"batch_size":4,

Configuration: max batch size = 64, delayed dynamic batching with pref batch size [12,36,64], req rate = 400


In [72]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [73]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 8490
    Avg send request rate: 383.47 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 1.88% of the requests were delayed. 
    Throughput: 399.74 infer/sec
    Avg latency: 6266 usec (standard deviation 5795 usec)
    p50 latency: 5176 usec
    p90 latency: 9258 usec
    p95 latency: 11625 usec
    p99 latency: 23996 usec
    Avg HTTP time: 6142 usec (send/recv 446 usec + response wait 5696 usec)
  Server: 
    Inference count: 8489
    Execution count: 6737
    Successful request count: 8489
    Avg request latency: 3509 usec (overhead 20 usec + queue 1258 usec + compute input 404 usec + co

In [74]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753103182490,"inference_count":28609,"execution_count":9125,"inference_stats":{"success":{"count":28609,"ns":19745185727493},"fail":{"count":0,"ns":0},"queue":{"count":28609,"ns":18236902472571},"compute_input":{"count":28609,"ns":170395545437},"compute_infer":{"count":28609,"ns":1327272100952},"compute_output":{"count":28609,"ns":2370355667},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":7129,"ns":1034979360},"compute_infer":{"count":7129,"ns":16153884392},"compute_output":{"count":7129,"ns":61536420}},{"batch_size":2,"compute_input":{"count":1334,"ns":879733669},"compute_infer":{"count":1334,"ns":3270741373},"compute_output":{"count":1334,"ns":14214995}},{"batch_size":3,"compute_input":{"count":251,"ns":245512640},"compute_infer":{"count":251,"ns":1367972981},"compute_output":{"count":251,"ns":3308509}},{"batch_siz

## Validating trace data 
running on gpu p100
with max batch size = 8 and dynamic batching {}

# Steps to activate tracing 

"wget https://raw.githubusercontent.com/triton-inference-server/server/main/docs/examples/python/trace_summary.py"

"nano summer2025/docker/docker-compose.yaml"

"command: tritonserver --trace-file=/tmp/trace.json --trace-rate=1 --trace-count=20000 --trace-log-frequency=4000 --trace-level=TIMESTAMPS --model-repository=/models"
 * trace-rate = 1 -> so that we trace every request
 * trace-count => the server will stop tracing after that many requests
 * trace-log-frequency => should match how many requests we send in the benchmark (request-rate*request-count) -> each trace file will contain this many request traces

Then build and force recreate the server:

"docker compose -f ~/summer2025/docker/docker-compose.yaml build triton_server"

"docker compose -f ~/summer2025/docker/docker-compose.yaml up triton_server --force-recreate -d"

To see the trace files logged:

"docker exec triton_server ls -alstr /tmp/"

To copy them to the jupyter interface: 

"docker cp triton_server:/tmp/trace.json.1 ."

"docker cp trace.json.1 jupyter:/home/jovyan/work/trace_r6_1.json"

In [3]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


Warmup requests

In [5]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f warmup.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1826
    Throughput: 97.7208 infer/sec
    Avg latency: 9809 usec (standard deviation 2874 usec)
    p50 latency: 8806 usec
    p90 latency: 13648 usec
    p95 latency: 14705 usec
    p99 latency: 16730 usec
    Avg HTTP time: 9740 usec (send/recv 646 usec + response wait 9094 usec)
  Server: 
    Inference count: 1826
    Execution count: 1603
    Successful request count: 1826
    Avg request latency: 7172 usec (overhead 40 usec + queue 1479 usec + compute input 200 usec + compute infer 5430 usec + compute output 22 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 97.7208 infer/sec, latency 9809 

In [6]:
#warmup stats
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753284888049,"inference_count":8199,"execution_count":2483,"inference_stats":{"success":{"count":8199,"ns":40989864628074},"fail":{"count":0,"ns":0},"queue":{"count":8199,"ns":40582998530353},"compute_input":{"count":8199,"ns":11172046876},"compute_infer":{"count":8199,"ns":394909627927},"compute_output":{"count":8199,"ns":209522527},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1513,"ns":209136266},"compute_infer":{"count":1513,"ns":14089962261},"compute_output":{"count":1513,"ns":32536301}},{"batch_size":2,"compute_input":{"count":143,"ns":50678806},"compute_infer":{"count":143,"ns":2157035596},"compute_output":{"count":143,"ns":3380178}},{"batch_size":3,"compute_input":{"count":36,"ns":19049209},"compute_infer":{"count":36,"ns":1620767943},"compute_output":{"count":36,"ns":905061}},{"batch_size":4,"compute_input"

In [7]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f test1.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1826
    Throughput: 97.7359 infer/sec
    Avg latency: 9632 usec (standard deviation 2677 usec)
    p50 latency: 8631 usec
    p90 latency: 13174 usec
    p95 latency: 14249 usec
    p99 latency: 15919 usec
    Avg HTTP time: 9571 usec (send/recv 643 usec + response wait 8928 usec)
  Server: 
    Inference count: 1826
    Execution count: 1619
    Successful request count: 1826
    Avg request latency: 7059 usec (overhead 38 usec + queue 1428 usec + compute input 191 usec + compute infer 5379 usec + compute output 21 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 97.7359 infer/sec, latency 9632 

In [8]:
#warmup + actual stats 
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753284915626,"inference_count":10035,"execution_count":4112,"inference_stats":{"success":{"count":10035,"ns":41002813712131},"fail":{"count":0,"ns":0},"queue":{"count":10035,"ns":40585606735187},"compute_input":{"count":10035,"ns":11523609573},"compute_infer":{"count":10035,"ns":404787679493},"compute_output":{"count":10035,"ns":249983778},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2974,"ns":408105813},"compute_infer":{"count":2974,"ns":21834765768},"compute_output":{"count":2974,"ns":64034832}},{"batch_size":2,"compute_input":{"count":276,"ns":96826728},"compute_infer":{"count":276,"ns":2916144215},"compute_output":{"count":276,"ns":6506773}},{"batch_size":3,"compute_input":{"count":67,"ns":35391659},"compute_infer":{"count":67,"ns":1793377302},"compute_output":{"count":67,"ns":1662175}},{"batch_size":4,"compute

In [13]:
#running again w pref batch
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats


{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [14]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f warmup2.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
Failed to obtain stable measurement.
Failed to obtain stable measurement.



: 2

In [15]:
#warmup stats
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753293076489,"inference_count":6414,"execution_count":2321,"inference_stats":{"success":{"count":6414,"ns":33272932392308},"fail":{"count":0,"ns":0},"queue":{"count":6414,"ns":33016517494022},"compute_input":{"count":6414,"ns":6755931265},"compute_infer":{"count":6414,"ns":249082381996},"compute_output":{"count":6414,"ns":157340137},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1568,"ns":211911061},"compute_infer":{"count":1568,"ns":14024730870},"compute_output":{"count":1568,"ns":33682883}},{"batch_size":2,"compute_input":{"count":195,"ns":75418444},"compute_infer":{"count":195,"ns":2562323156},"compute_output":{"count":195,"ns":4519905}},{"batch_size":4,"compute_input":{"count":1,"ns":832846},"compute_infer":{"count":1,"ns":10978057384},"compute_output":{"count":1,"ns":47681}},{"batch_size":6,"compute_input":{"cou

In [16]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f test2.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1853
    Throughput: 98.2817 infer/sec
    Avg latency: 8331 usec (standard deviation 3003 usec)
    p50 latency: 7725 usec
    p90 latency: 12269 usec
    p95 latency: 13559 usec
    p99 latency: 15973 usec
    Avg HTTP time: 8272 usec (send/recv 528 usec + response wait 7744 usec)
  Server: 
    Inference count: 1853
    Execution count: 1698
    Successful request count: 1853
    Avg request latency: 5901 usec (overhead 32 usec + queue 1225 usec + compute input 151 usec + compute infer 4474 usec + compute output 18 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 98.2817 infer/sec, latency 8331 

In [17]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753293120040,"inference_count":8864,"execution_count":4568,"inference_stats":{"success":{"count":8864,"ns":33287797631461},"fail":{"count":0,"ns":0},"queue":{"count":8864,"ns":33019598604421},"compute_input":{"count":8864,"ns":7134994250},"compute_infer":{"count":8864,"ns":260359940029},"compute_output":{"count":8864,"ns":205167903},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":3612,"ns":456385884},"compute_infer":{"count":3612,"ns":23196230795},"compute_output":{"count":3612,"ns":72561355}},{"batch_size":2,"compute_input":{"count":398,"ns":142712525},"compute_infer":{"count":398,"ns":3615352210},"compute_output":{"count":398,"ns":8994552}},{"batch_size":4,"compute_input":{"count":1,"ns":832846},"compute_infer":{"count":1,"ns":10978057384},"compute_output":{"count":1,"ns":47681}},{"batch_size":6,"compute_input":{"co

In [18]:
#pref batch size [4,6,8]
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [19]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f warmup3.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1884
    Throughput: 99.0408 infer/sec
    Avg latency: 10209 usec (standard deviation 3644 usec)
    p50 latency: 9281 usec
    p90 latency: 14579 usec
    p95 latency: 15803 usec
    p99 latency: 19313 usec
    Avg HTTP time: 10123 usec (send/recv 348 usec + response wait 9775 usec)
  Server: 
    Inference count: 1883
    Execution count: 1605
    Successful request count: 1883
    Avg request latency: 7656 usec (overhead 43 usec + queue 1683 usec + compute input 220 usec + compute infer 5687 usec + compute output 22 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 99.0408 infer/sec, latency 102

In [20]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753293491181,"inference_count":6490,"execution_count":2981,"inference_stats":{"success":{"count":6490,"ns":47781127326246},"fail":{"count":0,"ns":0},"queue":{"count":6490,"ns":47582379098624},"compute_input":{"count":6490,"ns":5483963104},"compute_infer":{"count":6490,"ns":192726286310},"compute_output":{"count":6490,"ns":152668126},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2229,"ns":289856693},"compute_infer":{"count":2229,"ns":17573682666},"compute_output":{"count":2229,"ns":46484324}},{"batch_size":2,"compute_input":{"count":223,"ns":79330170},"compute_infer":{"count":223,"ns":2838500024},"compute_output":{"count":223,"ns":4955638}},{"batch_size":3,"compute_input":{"count":61,"ns":33221129},"compute_infer":{"count":61,"ns":1965340200},"compute_output":{"count":61,"ns":1515198}},{"batch_size":4,"compute_input"

In [21]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f test3.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1974
    Throughput: 103.134 infer/sec
    Avg latency: 9804 usec (standard deviation 2917 usec)
    p50 latency: 9050 usec
    p90 latency: 13654 usec
    p95 latency: 14886 usec
    p99 latency: 16862 usec
    Avg HTTP time: 9728 usec (send/recv 707 usec + response wait 9021 usec)
  Server: 
    Inference count: 1973
    Execution count: 1718
    Successful request count: 1973
    Avg request latency: 7099 usec (overhead 38 usec + queue 1508 usec + compute input 206 usec + compute infer 5325 usec + compute output 22 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 103.134 infer/sec, latency 9804 

In [22]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753293613506,"inference_count":11518,"execution_count":7438,"inference_stats":{"success":{"count":11518,"ns":47815168245898},"fail":{"count":0,"ns":0},"queue":{"count":11518,"ns":47589373295083},"compute_input":{"count":11518,"ns":6438147826},"compute_infer":{"count":11518,"ns":218530214061},"compute_output":{"count":11518,"ns":259933261},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":6210,"ns":813941703},"compute_infer":{"count":6210,"ns":37516052401},"compute_output":{"count":6210,"ns":128542076}},{"batch_size":2,"compute_input":{"count":613,"ns":219732877},"compute_infer":{"count":613,"ns":4996655414},"compute_output":{"count":613,"ns":14046793}},{"batch_size":3,"compute_input":{"count":138,"ns":73964739},"compute_infer":{"count":138,"ns":2406085108},"compute_output":{"count":138,"ns":3462213}},{"batch_size":4,"co

In [27]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [28]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f warmup4.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 2.38071 infer/sec
    Avg latency: 29634089 usec (standard deviation 361206 usec)
    p50 latency: 29600211 usec
    p90 latency: 29948969 usec
    p95 latency: 40630295 usec
    p99 latency: 40653403 usec
    Avg HTTP time: 29634063 usec (send/recv 1121 usec + response wait 29632942 usec)
  Server: 
    Inference count: 100
    Execution count: 15
    Successful request count: 100
    Avg request latency: 29628855 usec (overhead 79 usec + queue 27428807 usec + compute input 3560 usec + compute infer 2196375 usec + compute output 33 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 2.38071 infer/sec, latency 29634089 usec


In [29]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753293896013,"inference_count":100,"execution_count":15,"inference_stats":{"success":{"count":100,"ns":2962885596531},"fail":{"count":0,"ns":0},"queue":{"count":100,"ns":2742880740648},"compute_input":{"count":100,"ns":356007066},"compute_infer":{"count":100,"ns":219637567246},"compute_output":{"count":100,"ns":3370119},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2,"ns":487532},"compute_infer":{"count":2,"ns":6092966770},"compute_output":{"count":2,"ns":77031}},{"batch_size":4,"compute_input":{"count":1,"ns":3655685},"compute_infer":{"count":1,"ns":12166605258},"compute_output":{"count":1,"ns":83165}},{"batch_size":6,"compute_input":{"count":1,"ns":2364691},"compute_infer":{"count":1,"ns":11594958442},"compute_output":{"count":1,"ns":41222}},{"batch_size":8,"compute_input":{"count":11,"ns":40838581},"compute_infer

In [30]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f test4.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1858
    Throughput: 98.3735 infer/sec
    Avg latency: 9389 usec (standard deviation 2724 usec)
    p50 latency: 8540 usec
    p90 latency: 13070 usec
    p95 latency: 14042 usec
    p99 latency: 15945 usec
    Avg HTTP time: 9339 usec (send/recv 282 usec + response wait 9057 usec)
  Server: 
    Inference count: 1858
    Execution count: 1646
    Successful request count: 1858
    Avg request latency: 7239 usec (overhead 38 usec + queue 1526 usec + compute input 200 usec + compute infer 5453 usec + compute output 21 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 98.3735 infer/sec, latency 9389 

In [31]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753293939116,"inference_count":2580,"execution_count":1944,"inference_stats":{"success":{"count":2580,"ns":3264886450841},"fail":{"count":0,"ns":0},"queue":{"count":2580,"ns":3022155067730},"compute_input":{"count":2580,"ns":1297221017},"compute_infer":{"count":2580,"ns":241262328884},"compute_output":{"count":2580,"ns":58108896},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1685,"ns":238265175},"compute_infer":{"count":1685,"ns":15182848892},"compute_output":{"count":1685,"ns":35662718}},{"batch_size":2,"compute_input":{"count":161,"ns":58256447},"compute_infer":{"count":161,"ns":2495348192},"compute_output":{"count":161,"ns":3676058}},{"batch_size":3,"compute_input":{"count":35,"ns":18815594},"compute_infer":{"count":35,"ns":1783154332},"compute_output":{"count":35,"ns":869080}},{"batch_size":4,"compute_input":{"c

In [32]:
#no preferred batch size again

In [33]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [34]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f warmup5.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 4.99941 infer/sec
    Avg latency: 16971407 usec (standard deviation 35033 usec)
    p50 latency: 17105211 usec
    p90 latency: 17454417 usec
    p95 latency: 17572137 usec
    p99 latency: 17911662 usec
    Avg HTTP time: 16971383 usec (send/recv 1179 usec + response wait 16970204 usec)
  Server: 
    Inference count: 100
    Execution count: 14
    Successful request count: 100
    Avg request latency: 16966111 usec (overhead 78 usec + queue 15934410 usec + compute input 3543 usec + compute infer 1028050 usec + compute output 29 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 4.99941 infer/sec, latency 16971407 usec


In [35]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753295662243,"inference_count":100,"execution_count":14,"inference_stats":{"success":{"count":100,"ns":1696611131937},"fail":{"count":0,"ns":0},"queue":{"count":100,"ns":1593441064655},"compute_input":{"count":100,"ns":354368027},"compute_infer":{"count":100,"ns":102805047414},"compute_output":{"count":100,"ns":2934776},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1,"ns":201777},"compute_infer":{"count":1,"ns":6107421442},"compute_output":{"count":1,"ns":41395}},{"batch_size":3,"compute_input":{"count":1,"ns":1171222},"compute_infer":{"count":1,"ns":1366800628},"compute_output":{"count":1,"ns":39519}},{"batch_size":8,"compute_input":{"count":12,"ns":43831573},"compute_infer":{"count":12,"ns":11574653011},"compute_output":{"count":12,"ns":346853}}],"memory_usage":[]}]}


In [36]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --async -f test5.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1913
    Throughput: 98.2746 infer/sec
    Avg latency: 8611 usec (standard deviation 4338 usec)
    p50 latency: 7783 usec
    p90 latency: 11780 usec
    p95 latency: 12743 usec
    p99 latency: 14507 usec
    Avg HTTP time: 8472 usec (send/recv 378 usec + response wait 8094 usec)
  Server: 
    Inference count: 1913
    Execution count: 1707
    Successful request count: 1913
    Avg request latency: 6583 usec (overhead 38 usec + queue 1295 usec + compute input 185 usec + compute infer 5042 usec + compute output 22 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 98.2746 infer/sec, latency 8611 

In [37]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753295781161,"inference_count":6447,"execution_count":2761,"inference_stats":{"success":{"count":6447,"ns":22457794376847},"fail":{"count":0,"ns":0},"queue":{"count":6447,"ns":22120414749368},"compute_input":{"count":6447,"ns":5642701850},"compute_infer":{"count":6447,"ns":331170939473},"compute_output":{"count":6447,"ns":160837025},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2040,"ns":272301718},"compute_infer":{"count":2040,"ns":16165157411},"compute_output":{"count":2040,"ns":44171971}},{"batch_size":2,"compute_input":{"count":190,"ns":65570702},"compute_infer":{"count":190,"ns":2443515664},"compute_output":{"count":190,"ns":4470660}},{"batch_size":3,"compute_input":{"count":38,"ns":20086742},"compute_infer":{"count":38,"ns":1563366525},"compute_output":{"count":38,"ns":919492}},{"batch_size":4,"compute_input":

In [38]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [39]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f test5.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 4.99942 infer/sec
    Avg latency: 17110654 usec (standard deviation 166907 usec)
    p50 latency: 17247599 usec
    p90 latency: 17588638 usec
    p95 latency: 17705618 usec
    p99 latency: 18087648 usec
    Avg HTTP time: 17110629 usec (send/recv 1103 usec + response wait 17109526 usec)
  Server: 
    Inference count: 100
    Execution count: 14
    Successful request count: 100
    Avg request latency: 17105296 usec (overhead 870 usec + queue 16062717 usec + compute input 3486 usec + compute infer 1038190 usec + compute output 32 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 4.99942 infer/sec, latency 17110654 usec


In [40]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753303676065,"inference_count":100,"execution_count":14,"inference_stats":{"success":{"count":100,"ns":1710529679371},"fail":{"count":0,"ns":0},"queue":{"count":100,"ns":1606271775410},"compute_input":{"count":100,"ns":348601016},"compute_infer":{"count":100,"ns":103819093713},"compute_output":{"count":100,"ns":3234785},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1,"ns":264762},"compute_infer":{"count":1,"ns":6124069788},"compute_output":{"count":1,"ns":53387}},{"batch_size":3,"compute_input":{"count":1,"ns":1167362},"compute_infer":{"count":1,"ns":1385809127},"compute_output":{"count":1,"ns":51514}},{"batch_size":8,"compute_input":{"count":12,"ns":43104271},"compute_infer":{"count":12,"ns":11692199568},"compute_output":{"count":12,"ns":378357}}],"memory_usage":[]}]}


In [41]:
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f test5.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 7.14183 infer/sec
    Avg latency: 1438164 usec (standard deviation 346303 usec)
    p50 latency: 978299 usec
    p90 latency: 1389107 usec
    p95 latency: 11935365 usec
    p99 latency: 11960684 usec
    Avg HTTP time: 1438144 usec (send/recv 1066 usec + response wait 1437078 usec)
  Server: 
    Inference count: 100
    Execution count: 36
    Successful request count: 100
    Avg request latency: 1434826 usec (overhead 91 usec + queue 724791 usec + compute input 1383 usec + compute infer 708535 usec + compute output 24 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 7.14183 infer/sec, latency 1438164 usec


In [42]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753303735143,"inference_count":200,"execution_count":50,"inference_stats":{"success":{"count":200,"ns":1854012378037},"fail":{"count":0,"ns":0},"queue":{"count":200,"ns":1678750974760},"compute_input":{"count":200,"ns":486958781},"compute_infer":{"count":200,"ns":174672673757},"compute_output":{"count":200,"ns":5712563},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":26,"ns":3556957},"compute_infer":{"count":26,"ns":6249219234},"compute_output":{"count":26,"ns":545581}},{"batch_size":2,"compute_input":{"count":1,"ns":400969},"compute_infer":{"count":1,"ns":1415964139},"compute_output":{"count":1,"ns":30077}},{"batch_size":3,"compute_input":{"count":2,"ns":1791274},"compute_infer":{"count":2,"ns":1392059735},"compute_output":{"count":2,"ns":78268}},{"batch_size":6,"compute_input":{"count":1,"ns":817952},"compute_infer"

In [43]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [44]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f warmup.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 4.9994 infer/sec
    Avg latency: 16972253 usec (standard deviation 77264 usec)
    p50 latency: 17107069 usec
    p90 latency: 17455706 usec
    p95 latency: 17572850 usec
    p99 latency: 17893647 usec
    Avg HTTP time: 16972231 usec (send/recv 1201 usec + response wait 16971030 usec)
  Server: 
    Inference count: 100
    Execution count: 14
    Successful request count: 100
    Avg request latency: 16966636 usec (overhead 117 usec + queue 15933943 usec + compute input 3503 usec + compute infer 1029039 usec + compute output 33 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 4.9994 infer/sec, latency 16972253 usec


In [45]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753385603340,"inference_count":100,"execution_count":14,"inference_stats":{"success":{"count":100,"ns":1696663658481},"fail":{"count":0,"ns":0},"queue":{"count":100,"ns":1593394386422},"compute_input":{"count":100,"ns":350341575},"compute_infer":{"count":100,"ns":102903911799},"compute_output":{"count":100,"ns":3366438},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1,"ns":212090},"compute_infer":{"count":1,"ns":6080978152},"compute_output":{"count":1,"ns":48958}},{"batch_size":3,"compute_input":{"count":1,"ns":1134847},"compute_infer":{"count":1,"ns":1332784861},"compute_output":{"count":1,"ns":30280}},{"batch_size":8,"compute_input":{"count":12,"ns":43340618},"compute_infer":{"count":12,"ns":11603072383},"compute_output":{"count":12,"ns":403330}}],"memory_usage":[]}]}


In [46]:
#trace1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f test1.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 7.69117 infer/sec
    Avg latency: 1283937 usec (standard deviation 250583 usec)
    p50 latency: 990078 usec
    p90 latency: 1330922 usec
    p95 latency: 1375752 usec
    p99 latency: 11472309 usec
    Avg HTTP time: 1283920 usec (send/recv 1001 usec + response wait 1282919 usec)
  Server: 
    Inference count: 100
    Execution count: 26
    Successful request count: 100
    Avg request latency: 1280534 usec (overhead 97 usec + queue 809796 usec + compute input 1477 usec + compute infer 469137 usec + compute output 26 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 7.69117 infer/sec, latency 1283937 usec


In [47]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753385628838,"inference_count":200,"execution_count":40,"inference_stats":{"success":{"count":200,"ns":1824717111925},"fail":{"count":0,"ns":0},"queue":{"count":200,"ns":1674374003698},"compute_input":{"count":200,"ns":498045969},"compute_infer":{"count":200,"ns":149817705417},"compute_output":{"count":200,"ns":5972787},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":15,"ns":1681736},"compute_infer":{"count":15,"ns":6136363160},"compute_output":{"count":15,"ns":294221}},{"batch_size":2,"compute_input":{"count":1,"ns":418426},"compute_infer":{"count":1,"ns":1370641283},"compute_output":{"count":1,"ns":30583}},{"batch_size":3,"compute_input":{"count":1,"ns":1134847},"compute_infer":{"count":1,"ns":1332784861},"compute_output":{"count":1,"ns":30280}},{"batch_size":4,"compute_input":{"count":1,"ns":549548},"compute_infer"

In [48]:
#trace2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f test5.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 49.97 infer/sec
    Avg latency: 9402 usec (standard deviation 2362 usec)
    p50 latency: 8366 usec
    p90 latency: 12634 usec
    p95 latency: 14491 usec
    p99 latency: 16976 usec
    Avg HTTP time: 9380 usec (send/recv 725 usec + response wait 8655 usec)
  Server: 
    Inference count: 100
    Execution count: 93
    Successful request count: 100
    Avg request latency: 6726 usec (overhead 54 usec + queue 1236 usec + compute input 175 usec + compute infer 5240 usec + compute output 21 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 49.97 infer/sec, latency 9402 usec


In [49]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753385640039,"inference_count":300,"execution_count":133,"inference_stats":{"success":{"count":300,"ns":1825389778063},"fail":{"count":0,"ns":0},"queue":{"count":300,"ns":1674497617261},"compute_input":{"count":300,"ns":515565838},"compute_infer":{"count":300,"ns":150341712379},"compute_output":{"count":300,"ns":8128744},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":103,"ns":13680260},"compute_infer":{"count":103,"ns":6588823437},"compute_output":{"count":103,"ns":2155697}},{"batch_size":2,"compute_input":{"count":4,"ns":1512177},"compute_infer":{"count":4,"ns":1388283269},"compute_output":{"count":4,"ns":99867}},{"batch_size":3,"compute_input":{"count":3,"ns":2246128},"compute_infer":{"count":3,"ns":1344872432},"compute_output":{"count":3,"ns":82251}},{"batch_size":4,"compute_input":{"count":1,"ns":549548},"compute

In [58]:
#changed config to have preferred batch size [2,4,6,8]

In [51]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [52]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f warmup2.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 4.99945 infer/sec
    Avg latency: 17954030 usec (standard deviation 393433 usec)
    p50 latency: 18314797 usec
    p90 latency: 18627641 usec
    p95 latency: 18722118 usec
    p99 latency: 18836072 usec
    Avg HTTP time: 17954007 usec (send/recv 1099 usec + response wait 17952908 usec)
  Server: 
    Inference count: 100
    Execution count: 15
    Successful request count: 100
    Avg request latency: 17948878 usec (overhead 116 usec + queue 16934253 usec + compute input 3557 usec + compute infer 1010923 usec + compute output 29 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 4.99945 infer/sec, latency 17954030 usec


In [53]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753385870254,"inference_count":100,"execution_count":15,"inference_stats":{"success":{"count":100,"ns":1794887833559},"fail":{"count":0,"ns":0},"queue":{"count":100,"ns":1693425319577},"compute_input":{"count":100,"ns":355710766},"compute_infer":{"count":100,"ns":101092316338},"compute_output":{"count":100,"ns":2918612},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2,"ns":497826},"compute_infer":{"count":2,"ns":6070832034},"compute_output":{"count":2,"ns":59824}},{"batch_size":2,"compute_input":{"count":1,"ns":2795766},"compute_infer":{"count":1,"ns":1337643612},"compute_output":{"count":1,"ns":35326}},{"batch_size":8,"compute_input":{"count":12,"ns":43702676},"compute_infer":{"count":12,"ns":11543274635},"compute_output":{"count":12,"ns":348517}}],"memory_usage":[]}]}


In [54]:
#trace1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f test1b.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 49.9726 infer/sec
    Avg latency: 9671 usec (standard deviation 2638 usec)
    p50 latency: 8467 usec
    p90 latency: 13405 usec
    p95 latency: 15342 usec
    p99 latency: 17407 usec
    Avg HTTP time: 9651 usec (send/recv 901 usec + response wait 8750 usec)
  Server: 
    Inference count: 100
    Execution count: 94
    Successful request count: 100
    Avg request latency: 6976 usec (overhead 52 usec + queue 1438 usec + compute input 164 usec + compute infer 5300 usec + compute output 21 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 49.9726 infer/sec, latency 9671 usec


In [55]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753385893125,"inference_count":200,"execution_count":109,"inference_stats":{"success":{"count":200,"ns":1795585500626},"fail":{"count":0,"ns":0},"queue":{"count":200,"ns":1693569157857},"compute_input":{"count":200,"ns":372179578},"compute_infer":{"count":200,"ns":101622403360},"compute_output":{"count":200,"ns":5036816},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":90,"ns":12507954},"compute_infer":{"count":90,"ns":6530475652},"compute_output":{"count":90,"ns":1909594}},{"batch_size":2,"compute_input":{"count":7,"ns":5025108},"compute_infer":{"count":7,"ns":1372865314},"compute_output":{"count":7,"ns":169543}},{"batch_size":8,"compute_input":{"count":12,"ns":43702676},"compute_infer":{"count":12,"ns":11543274635},"compute_output":{"count":12,"ns":348517}}],"memory_usage":[]}]}


In [56]:
#trace2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=100 --async -f test2b.csv

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 100 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 100
    Throughput: 49.968 infer/sec
    Avg latency: 9848 usec (standard deviation 2553 usec)
    p50 latency: 8563 usec
    p90 latency: 13420 usec
    p95 latency: 15137 usec
    p99 latency: 17198 usec
    Avg HTTP time: 9829 usec (send/recv 879 usec + response wait 8950 usec)
  Server: 
    Inference count: 100
    Execution count: 93
    Successful request count: 100
    Avg request latency: 7110 usec (overhead 53 usec + queue 1421 usec + compute input 174 usec + compute infer 5439 usec + compute output 22 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 49.968 infer/sec, latency 9848 usec


In [57]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753385911507,"inference_count":300,"execution_count":202,"inference_stats":{"success":{"count":300,"ns":1796296547014},"fail":{"count":0,"ns":0},"queue":{"count":300,"ns":1693711303049},"compute_input":{"count":300,"ns":389628140},"compute_infer":{"count":300,"ns":102166317411},"compute_output":{"count":300,"ns":7277699},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":176,"ns":24711518},"compute_infer":{"count":176,"ns":6992630121},"compute_output":{"count":176,"ns":3801687}},{"batch_size":2,"compute_input":{"count":14,"ns":7647607},"compute_infer":{"count":14,"ns":1413745105},"compute_output":{"count":14,"ns":343938}},{"batch_size":8,"compute_input":{"count":12,"ns":43702676},"compute_infer":{"count":12,"ns":11543274635},"compute_output":{"count":12,"ns":348517}}],"memory_usage":[]}]}


## Tracing 1000 requests for each run 
configurartion: dynamic_batching{preferred_batch_size: [2,4,6,8]}

and max batch size of 8

In [59]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [60]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=1000 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 1000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1000
    Throughput: 31.2471 infer/sec
    Avg latency: 14508056 usec (standard deviation 115165 usec)
    p50 latency: 14641283 usec
    p90 latency: 18008038 usec
    p95 latency: 18518815 usec
    p99 latency: 18888898 usec
    Avg HTTP time: 14508037 usec (send/recv 1961 usec + response wait 14506076 usec)
  Server: 
    Inference count: 1000
    Execution count: 128
    Successful request count: 1000
    Avg request latency: 14500333 usec (overhead 123 usec + queue 14346315 usec + compute input 3032 usec + compute infer 150838 usec + compute output 25 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 31.2471 infer/sec, latency 14508056 usec


In [61]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753390814741,"inference_count":1000,"execution_count":128,"inference_stats":{"success":{"count":1000,"ns":14500333193698},"fail":{"count":0,"ns":0},"queue":{"count":1000,"ns":14346315031233},"compute_input":{"count":1000,"ns":3032417764},"compute_infer":{"count":1000,"ns":150838047033},"compute_output":{"count":1000,"ns":25230890},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2,"ns":485778},"compute_infer":{"count":2,"ns":6242089007},"compute_output":{"count":2,"ns":61568}},{"batch_size":2,"compute_input":{"count":1,"ns":2233871},"compute_infer":{"count":1,"ns":1370154183},"compute_output":{"count":1,"ns":55887}},{"batch_size":4,"compute_input":{"count":1,"ns":597335},"compute_infer":{"count":1,"ns":11118874953},"compute_output":{"count":1,"ns":54437}},{"batch_size":8,"compute_input":{"count":124,"ns":378134363},"co

In [62]:
#trace 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=1000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 1000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1000
    Throughput: 90.8951 infer/sec
    Avg latency: 9820 usec (standard deviation 2499 usec)
    p50 latency: 8608 usec
    p90 latency: 13491 usec
    p95 latency: 14902 usec
    p99 latency: 17449 usec
    Avg HTTP time: 9802 usec (send/recv 877 usec + response wait 8925 usec)
  Server: 
    Inference count: 1000
    Execution count: 897
    Successful request count: 1000
    Avg request latency: 7099 usec (overhead 55 usec + queue 1586 usec + compute input 174 usec + compute infer 5262 usec + compute output 20 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 90.8951 infer/sec, latency 9820 usec


In [63]:
#trace 1 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753390836168,"inference_count":2000,"execution_count":1025,"inference_stats":{"success":{"count":2000,"ns":14507432591460},"fail":{"count":0,"ns":0},"queue":{"count":2000,"ns":14347901593967},"compute_input":{"count":2000,"ns":3207181982},"compute_infer":{"count":2000,"ns":156100862461},"compute_output":{"count":2000,"ns":46011236},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":796,"ns":106954970},"compute_infer":{"count":796,"ns":10377078035},"compute_output":{"count":796,"ns":16341984}},{"batch_size":2,"compute_input":{"count":104,"ns":36381384},"compute_infer":{"count":104,"ns":1934067383},"compute_output":{"count":104,"ns":2305852}},{"batch_size":4,"compute_input":{"count":1,"ns":597335},"compute_infer":{"count":1,"ns":11118874953},"compute_output":{"count":1,"ns":54437}},{"batch_size":8,"compute_input":{"count":

In [64]:
# trace 2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=1000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 1000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1000
    Throughput: 90.8925 infer/sec
    Avg latency: 9293 usec (standard deviation 2636 usec)
    p50 latency: 8450 usec
    p90 latency: 13065 usec
    p95 latency: 14287 usec
    p99 latency: 17194 usec
    Avg HTTP time: 9275 usec (send/recv 903 usec + response wait 8372 usec)
  Server: 
    Inference count: 1000
    Execution count: 906
    Successful request count: 1000
    Avg request latency: 6531 usec (overhead 52 usec + queue 1412 usec + compute input 162 usec + compute infer 4886 usec + compute output 19 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 90.8925 infer/sec, latency 9293 usec


In [65]:
#trace 2 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753390869138,"inference_count":3000,"execution_count":1931,"inference_stats":{"success":{"count":3000,"ns":14513963890111},"fail":{"count":0,"ns":0},"queue":{"count":3000,"ns":14349314034867},"compute_input":{"count":3000,"ns":3369494511},"compute_infer":{"count":3000,"ns":160987103151},"compute_output":{"count":3000,"ns":65246986},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1608,"ns":207443969},"compute_infer":{"count":1608,"ns":14286887583},"compute_output":{"count":1608,"ns":31733582}},{"batch_size":2,"compute_input":{"count":198,"ns":67293149},"compute_infer":{"count":198,"ns":2422282954},"compute_output":{"count":198,"ns":4227928}},{"batch_size":4,"compute_input":{"count":1,"ns":597335},"compute_infer":{"count":1,"ns":11118874953},"compute_output":{"count":1,"ns":54437}},{"batch_size":8,"compute_input":{"coun

In [66]:
#trace 3
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=1000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 1000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1000
    Throughput: 90.8921 infer/sec
    Avg latency: 9761 usec (standard deviation 2404 usec)
    p50 latency: 8651 usec
    p90 latency: 13405 usec
    p95 latency: 14665 usec
    p99 latency: 16781 usec
    Avg HTTP time: 9740 usec (send/recv 916 usec + response wait 8824 usec)
  Server: 
    Inference count: 1000
    Execution count: 897
    Successful request count: 1000
    Avg request latency: 7058 usec (overhead 54 usec + queue 1554 usec + compute input 178 usec + compute infer 5251 usec + compute output 20 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 90.8921 infer/sec, latency 9761 usec


In [67]:
#trace 3 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753390895747,"inference_count":4000,"execution_count":2828,"inference_stats":{"success":{"count":4000,"ns":14521022731608},"fail":{"count":0,"ns":0},"queue":{"count":4000,"ns":14350868479485},"compute_input":{"count":4000,"ns":3548285821},"compute_infer":{"count":4000,"ns":166238341441},"compute_output":{"count":4000,"ns":85809796},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2402,"ns":315418331},"compute_infer":{"count":2402,"ns":18415991355},"compute_output":{"count":2402,"ns":47830744}},{"batch_size":2,"compute_input":{"count":301,"ns":102701623},"compute_infer":{"count":301,"ns":2983350213},"compute_output":{"count":301,"ns":6460752}},{"batch_size":4,"compute_input":{"count":1,"ns":597335},"compute_infer":{"count":1,"ns":11118874953},"compute_output":{"count":1,"ns":54437}},{"batch_size":8,"compute_input":{"cou

## Again,Tracing 1000 reqs (diff config)

In [68]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=1000 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 1000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1000
    Throughput: 33.3299 infer/sec
    Avg latency: 12763067 usec (standard deviation 68599 usec)
    p50 latency: 12816264 usec
    p90 latency: 16308999 usec
    p95 latency: 16870201 usec
    p99 latency: 17305622 usec
    Avg HTTP time: 12763050 usec (send/recv 2048 usec + response wait 12761002 usec)
  Server: 
    Inference count: 1000
    Execution count: 126
    Successful request count: 1000
    Avg request latency: 12755110 usec (overhead 113 usec + queue 12574778 usec + compute input 2620 usec + compute infer 177574 usec + compute output 24 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 33.3299 infer/sec, latency 12763067 usec


In [69]:
#trace 1 before
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753394685467,"inference_count":1000,"execution_count":126,"inference_stats":{"success":{"count":1000,"ns":12755110426124},"fail":{"count":0,"ns":0},"queue":{"count":1000,"ns":12574778190252},"compute_input":{"count":1000,"ns":2620169620},"compute_infer":{"count":1000,"ns":177574808012},"compute_output":{"count":1000,"ns":24343195},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1,"ns":204995},"compute_infer":{"count":1,"ns":6124903839},"compute_output":{"count":1,"ns":39335}},{"batch_size":7,"compute_input":{"count":1,"ns":930655},"compute_infer":{"count":1,"ns":10825116483},"compute_output":{"count":1,"ns":80764}},{"batch_size":8,"compute_input":{"count":124,"ns":326681255},"compute_infer":{"count":124,"ns":11959261099},"compute_output":{"count":124,"ns":2967314}}],"memory_usage":[]}]}


In [70]:
#trace 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=1000 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 1000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1000
    Throughput: 62.4921 infer/sec
    Avg latency: 5020429 usec (standard deviation 89866 usec)
    p50 latency: 6132883 usec
    p90 latency: 9934853 usec
    p95 latency: 10480145 usec
    p99 latency: 10816323 usec
    Avg HTTP time: 5020412 usec (send/recv 1358 usec + response wait 5019054 usec)
  Server: 
    Inference count: 1000
    Execution count: 212
    Successful request count: 1000
    Avg request latency: 5015645 usec (overhead 105 usec + queue 4936338 usec + compute input 1411 usec + compute infer 77765 usec + compute output 25 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 62.4921 infer/sec, latency 5020429 usec


In [71]:
#trace 1 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753394716040,"inference_count":2000,"execution_count":338,"inference_stats":{"success":{"count":2000,"ns":17770755574919},"fail":{"count":0,"ns":0},"queue":{"count":2000,"ns":17511116872061},"compute_input":{"count":2000,"ns":4032051462},"compute_infer":{"count":2000,"ns":255340391508},"compute_output":{"count":2000,"ns":49494147},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":91,"ns":11957344},"compute_infer":{"count":91,"ns":6537761328},"compute_output":{"count":91,"ns":1768944}},{"batch_size":2,"compute_input":{"count":9,"ns":3203322},"compute_infer":{"count":9,"ns":1426715113},"compute_output":{"count":9,"ns":191884}},{"batch_size":3,"compute_input":{"count":2,"ns":1699953},"compute_infer":{"count":2,"ns":1409312981},"compute_output":{"count":2,"ns":46249}},{"batch_size":6,"compute_input":{"count":1,"ns":834177},

In [72]:
#trace 2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=1000 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 1000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1000
    Throughput: 38.4575 infer/sec
    Avg latency: 5480364 usec (standard deviation 133350 usec)
    p50 latency: 6240821 usec
    p90 latency: 10042599 usec
    p95 latency: 10450786 usec
    p99 latency: 10772807 usec
    Avg HTTP time: 5480347 usec (send/recv 1594 usec + response wait 5478753 usec)
  Server: 
    Inference count: 1000
    Execution count: 337
    Successful request count: 1000
    Avg request latency: 5474799 usec (overhead 98 usec + queue 5371256 usec + compute input 1135 usec + compute infer 102286 usec + compute output 24 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 38.4575 infer/sec, latency 5480364 usec


In [73]:
#trace 2 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753394748127,"inference_count":3000,"execution_count":675,"inference_stats":{"success":{"count":3000,"ns":23245555222570},"fail":{"count":0,"ns":0},"queue":{"count":3000,"ns":22882373516652},"compute_input":{"count":3000,"ns":5167107882},"compute_infer":{"count":3000,"ns":357626741918},"compute_output":{"count":3000,"ns":73572289},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":312,"ns":42867206},"compute_infer":{"count":312,"ns":7656420192},"compute_output":{"count":312,"ns":6610171}},{"batch_size":2,"compute_input":{"count":31,"ns":10837171},"compute_infer":{"count":31,"ns":1552322333},"compute_output":{"count":31,"ns":735258}},{"batch_size":3,"compute_input":{"count":4,"ns":2812121},"compute_infer":{"count":4,"ns":1420943813},"compute_output":{"count":4,"ns":99486}},{"batch_size":4,"compute_input":{"count":1,"ns":7

In [74]:
#trace 3
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 100 --request-count=1000 --async 

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 1000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 100 inference requests per second
  Client: 
    Request count: 1000
    Throughput: 90.8949 infer/sec
    Avg latency: 9475 usec (standard deviation 2136 usec)
    p50 latency: 8528 usec
    p90 latency: 12777 usec
    p95 latency: 13856 usec
    p99 latency: 15710 usec
    Avg HTTP time: 9456 usec (send/recv 924 usec + response wait 8532 usec)
  Server: 
    Inference count: 1000
    Execution count: 889
    Successful request count: 1000
    Avg request latency: 6741 usec (overhead 58 usec + queue 1310 usec + compute input 191 usec + compute infer 5160 usec + compute output 21 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 100, throughput: 90.8949 infer/sec, latency 9475 usec


In [75]:
#trace 3 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753394766929,"inference_count":4000,"execution_count":1564,"inference_stats":{"success":{"count":4000,"ns":23252297143420},"fail":{"count":0,"ns":0},"queue":{"count":4000,"ns":22883684176973},"compute_input":{"count":4000,"ns":5358342398},"compute_infer":{"count":4000,"ns":362787604474},"compute_output":{"count":4000,"ns":95445034},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1110,"ns":153316492},"compute_infer":{"count":1110,"ns":11703884109},"compute_output":{"count":1110,"ns":23691012}},{"batch_size":2,"compute_input":{"count":105,"ns":35887611},"compute_infer":{"count":105,"ns":1957226401},"compute_output":{"count":105,"ns":2444039}},{"batch_size":3,"compute_input":{"count":18,"ns":10132383},"compute_infer":{"count":18,"ns":1498679506},"compute_output":{"count":18,"ns":446320}},{"batch_size":4,"compute_input":{

# Tracing with higher arrival rates
configuration is: max_batch_size = 8
dynamic batching{}

Notes: 

I tried tracing 1000 requests at a rate of 500req/sec - none of the traces were good (weird compute times)

Tried 5000 requests at once ta a rate of 500req/sec - again none of the traces were good.

Now trying dynamic batching{preferred_batch_size: [2,4,6,8]}

In [85]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [86]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 147.029 infer/sec
    Avg latency: 15089075 usec (standard deviation 30683 usec)
    p50 latency: 15074148 usec
    p90 latency: 17290550 usec
    p95 latency: 17481610 usec
    p99 latency: 17664254 usec
    Avg HTTP time: 15089062 usec (send/recv 4393 usec + response wait 15084669 usec)
  Server: 
    Inference count: 5000
    Execution count: 626
    Successful request count: 5000
    Avg request latency: 15073496 usec (overhead 112 usec + queue 15030948 usec + compute input 1705 usec + compute infer 40706 usec + compute output 24 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 147.029 infer/sec, latency 15089075 usec


In [87]:
#trace 1 before
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753464098255,"inference_count":5000,"execution_count":626,"inference_stats":{"success":{"count":5000,"ns":75367483007984},"fail":{"count":0,"ns":0},"queue":{"count":5000,"ns":75154744592626},"compute_input":{"count":5000,"ns":8525440358},"compute_infer":{"count":5000,"ns":203532974514},"compute_output":{"count":5000,"ns":123782509},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1,"ns":197151},"compute_infer":{"count":1,"ns":6366942426},"compute_output":{"count":1,"ns":44498}},{"batch_size":7,"compute_input":{"count":1,"ns":933433},"compute_infer":{"count":1,"ns":10959311712},"compute_output":{"count":1,"ns":46333}},{"batch_size":8,"compute_input":{"count":624,"ns":1064838647},"compute_infer":{"count":624,"ns":15056356263},"compute_output":{"count":624,"ns":15426710}}],"memory_usage":[]}]}


In [88]:
#trace 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 277.682 infer/sec
    Avg latency: 9527921 usec (standard deviation 53095 usec)
    p50 latency: 9589921 usec
    p90 latency: 11522741 usec
    p95 latency: 11735834 usec
    p99 latency: 11926476 usec
    Avg HTTP time: 9527908 usec (send/recv 4264 usec + response wait 9523644 usec)
  Server: 
    Inference count: 5000
    Execution count: 629
    Successful request count: 5000
    Avg request latency: 9513105 usec (overhead 131 usec + queue 9495422 usec + compute input 1979 usec + compute infer 15545 usec + compute output 27 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 277.682 infer/sec, latency 9527921 usec


In [89]:
#trace 1 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753464593466,"inference_count":10000,"execution_count":1255,"inference_stats":{"success":{"count":10000,"ns":122933009845024},"fail":{"count":0,"ns":0},"queue":{"count":10000,"ns":122631858369587},"compute_input":{"count":10000,"ns":18422561375},"compute_infer":{"count":10000,"ns":281261181478},"compute_output":{"count":10000,"ns":261499124},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":5,"ns":592772},"compute_infer":{"count":5,"ns":6379687190},"compute_output":{"count":5,"ns":103953}},{"batch_size":4,"compute_input":{"count":1,"ns":656603},"compute_infer":{"count":1,"ns":11968152074},"compute_output":{"count":1,"ns":72260}},{"batch_size":7,"compute_input":{"count":1,"ns":933433},"compute_infer":{"count":1,"ns":10959311712},"compute_output":{"count":1,"ns":46333}},{"batch_size":8,"compute_input":{"count":1248,"ns":2

In [90]:
#trace 2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 277.661 infer/sec
    Avg latency: 7011558 usec (standard deviation 46026 usec)
    p50 latency: 8621781 usec
    p90 latency: 10608122 usec
    p95 latency: 10832902 usec
    p99 latency: 11023689 usec
    Avg HTTP time: 7011542 usec (send/recv 2803 usec + response wait 7008739 usec)
  Server: 
    Inference count: 5000
    Execution count: 636
    Successful request count: 5000
    Avg request latency: 7001210 usec (overhead 115 usec + queue 6979209 usec + compute input 1796 usec + compute infer 20064 usec + compute output 26 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 277.661 infer/sec, latency 7011558 usec


In [91]:
#trace 2 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753464793482,"inference_count":15000,"execution_count":1891,"inference_stats":{"success":{"count":15000,"ns":157939061920004},"fail":{"count":0,"ns":0},"queue":{"count":15000,"ns":157527904249504},"compute_input":{"count":15000,"ns":27405046923},"compute_infer":{"count":15000,"ns":381582093571},"compute_output":{"count":15000,"ns":392937042},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":15,"ns":1572270},"compute_infer":{"count":15,"ns":6414898345},"compute_output":{"count":15,"ns":252835}},{"batch_size":3,"compute_input":{"count":1,"ns":470393},"compute_infer":{"count":1,"ns":1443257964},"compute_output":{"count":1,"ns":29426}},{"batch_size":4,"compute_input":{"count":3,"ns":1851846},"compute_infer":{"count":3,"ns":11975651931},"compute_output":{"count":3,"ns":106557}},{"batch_size":6,"compute_input":{"count":2,"ns"

In [92]:
#trace 3
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 277.639 infer/sec
    Avg latency: 6817673 usec (standard deviation 56820 usec)
    p50 latency: 8426119 usec
    p90 latency: 10529944 usec
    p95 latency: 10755823 usec
    p99 latency: 10968806 usec
    Avg HTTP time: 6817657 usec (send/recv 2738 usec + response wait 6814919 usec)
  Server: 
    Inference count: 5000
    Execution count: 670
    Successful request count: 5000
    Avg request latency: 6807575 usec (overhead 115 usec + queue 6788368 usec + compute input 1587 usec + compute infer 17476 usec + compute output 28 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 277.639 infer/sec, latency 6817673 usec


In [93]:
#trace 3 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753464866757,"inference_count":20000,"execution_count":2561,"inference_stats":{"success":{"count":20000,"ns":191976940943182},"fail":{"count":0,"ns":0},"queue":{"count":20000,"ns":191469746327385},"compute_input":{"count":20000,"ns":35344501636},"compute_infer":{"count":20000,"ns":468964679052},"compute_output":{"count":20000,"ns":533162380},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":46,"ns":4451313},"compute_infer":{"count":46,"ns":6515243229},"compute_output":{"count":46,"ns":653289}},{"batch_size":2,"compute_input":{"count":12,"ns":2883269},"compute_infer":{"count":12,"ns":1428524367},"compute_output":{"count":12,"ns":182198}},{"batch_size":3,"compute_input":{"count":10,"ns":3828919},"compute_infer":{"count":10,"ns":1473317644},"compute_output":{"count":10,"ns":168235}},{"batch_size":4,"compute_input":{"count"

In [94]:
#new config
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [95]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 131.557 infer/sec
    Avg latency: 17767807 usec (standard deviation 37573 usec)
    p50 latency: 17463988 usec
    p90 latency: 19672712 usec
    p95 latency: 19785800 usec
    p99 latency: 19899444 usec
    Avg HTTP time: 17767791 usec (send/recv 4413 usec + response wait 17763378 usec)
  Server: 
    Inference count: 5000
    Execution count: 628
    Successful request count: 5000
    Avg request latency: 17751822 usec (overhead 129 usec + queue 17712309 usec + compute input 3833 usec + compute infer 35525 usec + compute output 26 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 131.557 infer/sec, latency 17767807 usec


In [96]:
#trace 1 before
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753467176180,"inference_count":5000,"execution_count":628,"inference_stats":{"success":{"count":5000,"ns":88759110619411},"fail":{"count":0,"ns":0},"queue":{"count":5000,"ns":88561548570166},"compute_input":{"count":5000,"ns":19165217278},"compute_infer":{"count":5000,"ns":177628356462},"compute_output":{"count":5000,"ns":130621370},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2,"ns":1147860},"compute_infer":{"count":2,"ns":6606983252},"compute_output":{"count":2,"ns":63160}},{"batch_size":2,"compute_input":{"count":1,"ns":2303487},"compute_infer":{"count":1,"ns":1600473441},"compute_output":{"count":1,"ns":60963}},{"batch_size":4,"compute_input":{"count":1,"ns":2778657},"compute_infer":{"count":1,"ns":11085021320},"compute_output":{"count":1,"ns":67667}},{"batch_size":8,"compute_input":{"count":624,"ns":2393543477

In [97]:
#trace 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 293.978 infer/sec
    Avg latency: 7886885 usec (standard deviation 45070 usec)
    p50 latency: 8730204 usec
    p90 latency: 10923841 usec
    p95 latency: 11187145 usec
    p99 latency: 11399991 usec
    Avg HTTP time: 7886870 usec (send/recv 3376 usec + response wait 7883494 usec)
  Server: 
    Inference count: 5000
    Execution count: 937
    Successful request count: 5000
    Avg request latency: 7874769 usec (overhead 107 usec + queue 7854068 usec + compute input 1328 usec + compute infer 19243 usec + compute output 23 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 293.978 infer/sec, latency 7886885 usec


In [98]:
#trace 1 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753467243284,"inference_count":10000,"execution_count":1565,"inference_stats":{"success":{"count":10000,"ns":128132956684536},"fail":{"count":0,"ns":0},"queue":{"count":10000,"ns":127831891461496},"compute_input":{"count":10000,"ns":25806928612},"compute_infer":{"count":10000,"ns":273845609357},"compute_output":{"count":10000,"ns":246307381},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":138,"ns":16770904},"compute_infer":{"count":138,"ns":7014989975},"compute_output":{"count":138,"ns":1723371}},{"batch_size":2,"compute_input":{"count":244,"ns":73727211},"compute_infer":{"count":244,"ns":2318842740},"compute_output":{"count":244,"ns":3378703}},{"batch_size":4,"compute_input":{"count":22,"ns":15125304},"compute_infer":{"count":22,"ns":11160310517},"compute_output":{"count":22,"ns":415424}},{"batch_size":6,"compute_inp

In [99]:
#trace 2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 454.191 infer/sec
    Avg latency: 9254 usec (standard deviation 2754 usec)
    p50 latency: 8921 usec
    p90 latency: 12932 usec
    p95 latency: 14172 usec
    p99 latency: 16359 usec
    Avg HTTP time: 9232 usec (send/recv 747 usec + response wait 8485 usec)
  Server: 
    Inference count: 5000
    Execution count: 2868
    Successful request count: 5000
    Avg request latency: 6429 usec (overhead 43 usec + queue 3096 usec + compute input 289 usec + compute infer 2987 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 454.191 infer/sec, latency 9254 usec


In [100]:
#trace 2 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753467288575,"inference_count":15000,"execution_count":4433,"inference_stats":{"success":{"count":15000,"ns":128165104950885},"fail":{"count":0,"ns":0},"queue":{"count":15000,"ns":127847372108964},"compute_input":{"count":15000,"ns":27253295966},"compute_infer":{"count":15000,"ns":288782364667},"compute_output":{"count":15000,"ns":314594605},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1156,"ns":121098722},"compute_infer":{"count":1156,"ns":9988051059},"compute_output":{"count":1156,"ns":14084213}},{"batch_size":2,"compute_input":{"count":1965,"ns":576677546},"compute_infer":{"count":1965,"ns":7293071279},"compute_output":{"count":1965,"ns":26748989}},{"batch_size":4,"compute_input":{"count":139,"ns":82366385},"compute_infer":{"count":139,"ns":11578946684},"compute_output":{"count":139,"ns":2344678}},{"batch_size":

In [101]:
#trace 3
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 454.311 infer/sec
    Avg latency: 9313 usec (standard deviation 2871 usec)
    p50 latency: 8937 usec
    p90 latency: 13132 usec
    p95 latency: 14554 usec
    p99 latency: 17717 usec
    Avg HTTP time: 9290 usec (send/recv 799 usec + response wait 8491 usec)
  Server: 
    Inference count: 5000
    Execution count: 2861
    Successful request count: 5000
    Avg request latency: 6350 usec (overhead 42 usec + queue 3039 usec + compute input 287 usec + compute infer 2968 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 454.311 infer/sec, latency 9313 usec


In [102]:
#trace 3 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753467322837,"inference_count":20000,"execution_count":7294,"inference_stats":{"success":{"count":20000,"ns":128196858478453},"fail":{"count":0,"ns":0},"queue":{"count":20000,"ns":127862570733062},"compute_input":{"count":20000,"ns":28689246899},"compute_infer":{"count":20000,"ns":303624785916},"compute_output":{"count":20000,"ns":382851629},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2122,"ns":223617085},"compute_infer":{"count":2122,"ns":12811949708},"compute_output":{"count":2122,"ns":25705689}},{"batch_size":2,"compute_input":{"count":3751,"ns":1096624732},"compute_infer":{"count":3751,"ns":12427010411},"compute_output":{"count":3751,"ns":51143921}},{"batch_size":4,"compute_input":{"count":237,"ns":138732312},"compute_infer":{"count":237,"ns":11928894928},"compute_output":{"count":237,"ns":3944140}},{"batch_si

## Trying lower request rates again

In [103]:

curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [104]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 300 --request-count=3000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 3000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 300 inference requests per second
  Client: 
    Request count: 3000
    Throughput: 81.0658 infer/sec
    Avg latency: 17266183 usec (standard deviation 65480 usec)
    p50 latency: 17243064 usec
    p90 latency: 20230551 usec
    p95 latency: 20504186 usec
    p99 latency: 20813634 usec
    Avg HTTP time: 17266167 usec (send/recv 2901 usec + response wait 17263266 usec)
  Server: 
    Inference count: 3000
    Execution count: 378
    Successful request count: 3000
    Avg request latency: 17253910 usec (overhead 133 usec + queue 17191766 usec + compute input 2008 usec + compute infer 59977 usec + compute output 24 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 300, throughput: 81.0658 infer/sec, latency 17266183 usec


In [105]:
#trace 1 before
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753475500652,"inference_count":3000,"execution_count":378,"inference_stats":{"success":{"count":3000,"ns":51761732051447},"fail":{"count":0,"ns":0},"queue":{"count":3000,"ns":51575300849757},"compute_input":{"count":3000,"ns":6026343242},"compute_infer":{"count":3000,"ns":179932561973},"compute_output":{"count":3000,"ns":74628099},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2,"ns":589568},"compute_infer":{"count":2,"ns":6356364815},"compute_output":{"count":2,"ns":52617}},{"batch_size":2,"compute_input":{"count":1,"ns":1992081},"compute_infer":{"count":1,"ns":1577831565},"compute_output":{"count":1,"ns":55679}},{"batch_size":4,"compute_input":{"count":1,"ns":1332114},"compute_infer":{"count":1,"ns":12225084619},"compute_output":{"count":1,"ns":66769}},{"batch_size":8,"compute_input":{"count":374,"ns":752055132},"c

In [106]:
#trace 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 300 --request-count=3000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 3000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 300 inference requests per second
  Client: 
    Request count: 3000
    Throughput: 272.667 infer/sec
    Avg latency: 9504 usec (standard deviation 2925 usec)
    p50 latency: 9022 usec
    p90 latency: 13557 usec
    p95 latency: 15196 usec
    p99 latency: 18194 usec
    Avg HTTP time: 9482 usec (send/recv 868 usec + response wait 8614 usec)
  Server: 
    Inference count: 3000
    Execution count: 2178
    Successful request count: 3000
    Avg request latency: 5899 usec (overhead 38 usec + queue 2193 usec + compute input 177 usec + compute infer 3477 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 300, throughput: 272.667 infer/sec, latency 9504 usec


In [107]:
#trace 1 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753475519785,"inference_count":6000,"execution_count":2556,"inference_stats":{"success":{"count":6000,"ns":51779430484774},"fail":{"count":0,"ns":0},"queue":{"count":6000,"ns":51581882583386},"compute_input":{"count":6000,"ns":6558374206},"compute_infer":{"count":6000,"ns":190366284065},"compute_output":{"count":6000,"ns":115293914},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1376,"ns":105648254},"compute_infer":{"count":1376,"ns":11063543417},"compute_output":{"count":1376,"ns":17585638}},{"batch_size":2,"compute_input":{"count":796,"ns":204706550},"compute_infer":{"count":796,"ns":4372362506},"compute_output":{"count":796,"ns":11319242}},{"batch_size":4,"compute_input":{"count":10,"ns":6717949},"compute_infer":{"count":10,"ns":12259455021},"compute_output":{"count":10,"ns":218186}},{"batch_size":8,"compute_input

In [108]:
#trace 2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 300 --request-count=3000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 3000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 300 inference requests per second
  Client: 
    Request count: 3000
    Throughput: 272.65 infer/sec
    Avg latency: 9344 usec (standard deviation 2824 usec)
    p50 latency: 8905 usec
    p90 latency: 13149 usec
    p95 latency: 14721 usec
    p99 latency: 18117 usec
    Avg HTTP time: 9322 usec (send/recv 846 usec + response wait 8476 usec)
  Server: 
    Inference count: 3000
    Execution count: 2185
    Successful request count: 3000
    Avg request latency: 5763 usec (overhead 37 usec + queue 2087 usec + compute input 174 usec + compute infer 3450 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 300, throughput: 272.65 infer/sec, latency 9344 usec


In [109]:
#trace 2 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753475551160,"inference_count":9000,"execution_count":4741,"inference_stats":{"success":{"count":9000,"ns":51796720462719},"fail":{"count":0,"ns":0},"queue":{"count":9000,"ns":51588144905276},"compute_input":{"count":9000,"ns":7082974624},"compute_infer":{"count":9000,"ns":200718725235},"compute_output":{"count":9000,"ns":155784210},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2760,"ns":211142328},"compute_infer":{"count":2760,"ns":15781372931},"compute_output":{"count":2760,"ns":35223938}},{"batch_size":2,"compute_input":{"count":1590,"ns":405863478},"compute_infer":{"count":1590,"ns":7137954354},"compute_output":{"count":1590,"ns":22515800}},{"batch_size":4,"compute_input":{"count":17,"ns":10916071},"compute_infer":{"count":17,"ns":12285312011},"compute_output":{"count":17,"ns":332906}},{"batch_size":8,"compute_i

In [110]:
#trace 3
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 300 --request-count=3000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 3000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 300 inference requests per second
  Client: 
    Request count: 3000
    Throughput: 272.686 infer/sec
    Avg latency: 9006 usec (standard deviation 2644 usec)
    p50 latency: 8606 usec
    p90 latency: 12590 usec
    p95 latency: 14149 usec
    p99 latency: 16909 usec
    Avg HTTP time: 8983 usec (send/recv 854 usec + response wait 8129 usec)
  Server: 
    Inference count: 3000
    Execution count: 2173
    Successful request count: 3000
    Avg request latency: 5794 usec (overhead 38 usec + queue 2097 usec + compute input 176 usec + compute infer 3468 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 300, throughput: 272.686 infer/sec, latency 9006 usec


In [111]:
#trace 3 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753475573350,"inference_count":12000,"execution_count":6914,"inference_stats":{"success":{"count":12000,"ns":51814102684129},"fail":{"count":0,"ns":0},"queue":{"count":12000,"ns":51594437967138},"compute_input":{"count":12000,"ns":7613059233},"compute_infer":{"count":12000,"ns":211125631496},"compute_output":{"count":12000,"ns":195873827},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":4124,"ns":314928529},"compute_infer":{"count":4124,"ns":20453778834},"compute_output":{"count":4124,"ns":52519521}},{"batch_size":2,"compute_input":{"count":2390,"ns":608528042},"compute_infer":{"count":2390,"ns":9938419891},"compute_output":{"count":2390,"ns":33620543}},{"batch_size":4,"compute_input":{"count":26,"ns":16158391},"compute_infer":{"count":26,"ns":12318704332},"compute_output":{"count":26,"ns":479043}},{"batch_size":8,"com

In [112]:
#increasing arrival rate to 400

In [113]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --request-count=4000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 4000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 4000
    Throughput: 114.271 infer/sec
    Avg latency: 16608644 usec (standard deviation 37765 usec)
    p50 latency: 16636859 usec
    p90 latency: 19256465 usec
    p95 latency: 19479862 usec
    p99 latency: 19712711 usec
    Avg HTTP time: 16608630 usec (send/recv 3557 usec + response wait 16605073 usec)
  Server: 
    Inference count: 4000
    Execution count: 503
    Successful request count: 4000
    Avg request latency: 16595058 usec (overhead 112 usec + queue 16550148 usec + compute input 1439 usec + compute infer 43333 usec + compute output 25 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 400, throughput: 114.271 infer/sec, latency 16608644 usec


In [114]:
#trace 1 before
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753476409050,"inference_count":4000,"execution_count":503,"inference_stats":{"success":{"count":4000,"ns":66380232630900},"fail":{"count":0,"ns":0},"queue":{"count":4000,"ns":66200594651556},"compute_input":{"count":4000,"ns":5756446597},"compute_infer":{"count":4000,"ns":173334765771},"compute_output":{"count":4000,"ns":101323719},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2,"ns":387653},"compute_infer":{"count":2,"ns":6368975407},"compute_output":{"count":2,"ns":62989}},{"batch_size":2,"compute_input":{"count":1,"ns":3004264},"compute_infer":{"count":1,"ns":1414159078},"compute_output":{"count":1,"ns":29585}},{"batch_size":4,"compute_input":{"count":1,"ns":554324},"compute_infer":{"count":1,"ns":11092682422},"compute_output":{"count":1,"ns":53152}},{"batch_size":8,"compute_input":{"count":499,"ns":718479140},"c

In [115]:
#trace 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --request-count=4000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 4000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 4000
    Throughput: 190.392 infer/sec
    Avg latency: 615216 usec (standard deviation 36824 usec)
    p50 latency: 8479 usec
    p90 latency: 13472 usec
    p95 latency: 10836850 usec
    p99 latency: 11123822 usec
    Avg HTTP time: 615195 usec (send/recv 834 usec + response wait 614361 usec)
  Server: 
    Inference count: 4000
    Execution count: 2571
    Successful request count: 4000
    Avg request latency: 611738 usec (overhead 42 usec + queue 591609 usec + compute input 250 usec + compute infer 19823 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 400, throughput: 190.392 infer/sec, latency 615216 usec


In [116]:
#trace 1 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753476454472,"inference_count":8000,"execution_count":3074,"inference_stats":{"success":{"count":8000,"ns":68827186431760},"fail":{"count":0,"ns":0},"queue":{"count":8000,"ns":68567033457177},"compute_input":{"count":8000,"ns":6757723401},"compute_infer":{"count":8000,"ns":252630187663},"compute_output":{"count":8000,"ns":154295249},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1356,"ns":99961421},"compute_infer":{"count":1356,"ns":10293558765},"compute_output":{"count":1356,"ns":16068155}},{"batch_size":2,"compute_input":{"count":1165,"ns":281967551},"compute_infer":{"count":1165,"ns":4809024927},"compute_output":{"count":1165,"ns":15110541}},{"batch_size":4,"compute_input":{"count":27,"ns":15035927},"compute_infer":{"count":27,"ns":11185705489},"compute_output":{"count":27,"ns":468016}},{"batch_size":6,"compute_in

In [117]:
#trace 2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --request-count=4000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 4000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 4000
    Throughput: 363.545 infer/sec
    Avg latency: 8872 usec (standard deviation 2753 usec)
    p50 latency: 8455 usec
    p90 latency: 12513 usec
    p95 latency: 14018 usec
    p99 latency: 17098 usec
    Avg HTTP time: 8849 usec (send/recv 829 usec + response wait 8020 usec)
  Server: 
    Inference count: 4000
    Execution count: 2656
    Successful request count: 4000
    Avg request latency: 5546 usec (overhead 37 usec + queue 2273 usec + compute input 200 usec + compute infer 3021 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 400, throughput: 363.545 infer/sec, latency 8872 usec


In [118]:
#trace 2 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753476503237,"inference_count":12000,"execution_count":5730,"inference_stats":{"success":{"count":12000,"ns":68849374045556},"fail":{"count":0,"ns":0},"queue":{"count":12000,"ns":68576126469299},"compute_input":{"count":12000,"ns":7561447522},"compute_infer":{"count":12000,"ns":264717903106},"compute_output":{"count":12000,"ns":207704387},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2740,"ns":204662524},"compute_infer":{"count":2740,"ns":14410697988},"compute_output":{"count":2740,"ns":33175223}},{"batch_size":2,"compute_input":{"count":2404,"ns":586955974},"compute_infer":{"count":2404,"ns":8535347857},"compute_output":{"count":2404,"ns":32087073}},{"batch_size":4,"compute_input":{"count":57,"ns":32556627},"compute_infer":{"count":57,"ns":11293860374},"compute_output":{"count":57,"ns":967306}},{"batch_size":6,"com

In [119]:
#trace 3
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 400 --request-count=4000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 4000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 400 inference requests per second
  Client: 
    Request count: 4000
    Throughput: 363.574 infer/sec
    Avg latency: 8830 usec (standard deviation 2722 usec)
    p50 latency: 8425 usec
    p90 latency: 12537 usec
    p95 latency: 14196 usec
    p99 latency: 16655 usec
    Avg HTTP time: 8807 usec (send/recv 815 usec + response wait 7992 usec)
  Server: 
    Inference count: 4000
    Execution count: 2677
    Successful request count: 4000
    Avg request latency: 5405 usec (overhead 37 usec + queue 2206 usec + compute input 190 usec + compute infer 2958 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 400, throughput: 363.574 infer/sec, latency 8830 usec


In [120]:
#trace 3 after
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753476527634,"inference_count":16000,"execution_count":8407,"inference_stats":{"success":{"count":16000,"ns":68870995066426},"fail":{"count":0,"ns":0},"queue":{"count":16000,"ns":68584951090921},"compute_input":{"count":16000,"ns":8324227922},"compute_infer":{"count":16000,"ns":276551715862},"compute_output":{"count":16000,"ns":260142442},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":4156,"ns":309487084},"compute_infer":{"count":4156,"ns":18550042450},"compute_output":{"count":4156,"ns":50453060}},{"batch_size":2,"compute_input":{"count":3634,"ns":881728664},"compute_infer":{"count":3634,"ns":12160053006},"compute_output":{"count":3634,"ns":48602916}},{"batch_size":4,"compute_input":{"count":88,"ns":49659242},"compute_infer":{"count":88,"ns":11405124873},"compute_output":{"count":88,"ns":1499439}},{"batch_size":6,"c

## Test 1: Does the issue persist without tracing

Keep the request_count

In [121]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [122]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 142.825 infer/sec
    Avg latency: 16297592 usec (standard deviation 49383 usec)
    p50 latency: 16277254 usec
    p90 latency: 18526092 usec
    p95 latency: 18724254 usec
    p99 latency: 18905956 usec
    Avg HTTP time: 16297578 usec (send/recv 4434 usec + response wait 16293144 usec)
  Server: 
    Inference count: 5000
    Execution count: 628
    Successful request count: 5000
    Avg request latency: 16281607 usec (overhead 75 usec + queue 16245231 usec + compute input 1797 usec + compute infer 34481 usec + compute output 22 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 142.825 infer/sec, latency 16297592 usec


In [123]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753723946420,"inference_count":5000,"execution_count":628,"inference_stats":{"success":{"count":5000,"ns":81408039854130},"fail":{"count":0,"ns":0},"queue":{"count":5000,"ns":81226156977842},"compute_input":{"count":5000,"ns":8989887682},"compute_infer":{"count":5000,"ns":172408621163},"compute_output":{"count":5000,"ns":110472397},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2,"ns":582834},"compute_infer":{"count":2,"ns":6274627249},"compute_output":{"count":2,"ns":42613}},{"batch_size":2,"compute_input":{"count":1,"ns":2078478},"compute_infer":{"count":1,"ns":1397664525},"compute_output":{"count":1,"ns":25538}},{"batch_size":4,"compute_input":{"count":1,"ns":1558913},"compute_infer":{"count":1,"ns":10850009656},"compute_output":{"count":1,"ns":31133}},{"batch_size":8,"compute_input":{"count":624,"ns":1122364030},

In [124]:
#test 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 293.976 infer/sec
    Avg latency: 8190749 usec (standard deviation 53765 usec)
    p50 latency: 8501317 usec
    p90 latency: 10678551 usec
    p95 latency: 10932736 usec
    p99 latency: 11156424 usec
    Avg HTTP time: 8190736 usec (send/recv 3852 usec + response wait 8186884 usec)
  Server: 
    Inference count: 5000
    Execution count: 748
    Successful request count: 5000
    Avg request latency: 8176949 usec (overhead 75 usec + queue 8156205 usec + compute input 1449 usec + compute infer 19194 usec + compute output 25 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 293.976 infer/sec, latency 8190749 usec


In [125]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753724397180,"inference_count":10000,"execution_count":1376,"inference_stats":{"success":{"count":10000,"ns":122292785334737},"fail":{"count":0,"ns":0},"queue":{"count":10000,"ns":122007184344914},"compute_input":{"count":10000,"ns":16236279697},"compute_infer":{"count":10000,"ns":268383498800},"compute_output":{"count":10000,"ns":238696191},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":52,"ns":4524403},"compute_infer":{"count":52,"ns":6428722198},"compute_output":{"count":52,"ns":692247}},{"batch_size":2,"compute_input":{"count":101,"ns":31467865},"compute_infer":{"count":101,"ns":1706791802},"compute_output":{"count":101,"ns":1477303}},{"batch_size":4,"compute_input":{"count":9,"ns":6974146},"compute_infer":{"count":9,"ns":10879447409},"compute_output":{"count":9,"ns":164134}},{"batch_size":6,"compute_input":{"cou

In [126]:
#test 2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 454.39 infer/sec
    Avg latency: 8536 usec (standard deviation 2717 usec)
    p50 latency: 8072 usec
    p90 latency: 12123 usec
    p95 latency: 13529 usec
    p99 latency: 16863 usec
    Avg HTTP time: 8513 usec (send/recv 806 usec + response wait 7707 usec)
  Server: 
    Inference count: 5000
    Execution count: 2958
    Successful request count: 5000
    Avg request latency: 5919 usec (overhead 26 usec + queue 2761 usec + compute input 236 usec + compute infer 2881 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 454.39 infer/sec, latency 8536 usec


In [127]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753724497418,"inference_count":15000,"execution_count":4334,"inference_stats":{"success":{"count":15000,"ns":122322382356510},"fail":{"count":0,"ns":0},"queue":{"count":15000,"ns":122020992101555},"compute_input":{"count":15000,"ns":17420580660},"compute_infer":{"count":15000,"ns":282793481689},"compute_output":{"count":15000,"ns":304560694},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":1176,"ns":83944606},"compute_infer":{"count":1176,"ns":9621116999},"compute_output":{"count":1176,"ns":14003898}},{"batch_size":2,"compute_input":{"count":1843,"ns":457867752},"compute_infer":{"count":1843,"ns":6567616327},"compute_output":{"count":1843,"ns":24390930}},{"batch_size":4,"compute_input":{"count":91,"ns":53625909},"compute_infer":{"count":91,"ns":11172707142},"compute_output":{"count":91,"ns":1507976}},{"batch_size":6,"c

In [128]:
#test 3
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --request-count=5000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 5000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 5000
    Throughput: 454.389 infer/sec
    Avg latency: 8816 usec (standard deviation 2728 usec)
    p50 latency: 8438 usec
    p90 latency: 12482 usec
    p95 latency: 13792 usec
    p99 latency: 16795 usec
    Avg HTTP time: 8792 usec (send/recv 752 usec + response wait 8040 usec)
  Server: 
    Inference count: 5000
    Execution count: 2946
    Successful request count: 5000
    Avg request latency: 6014 usec (overhead 27 usec + queue 2827 usec + compute input 237 usec + compute infer 2910 usec + compute output 13 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 500, throughput: 454.389 infer/sec, latency 8816 usec


In [129]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753724581790,"inference_count":20000,"execution_count":7280,"inference_stats":{"success":{"count":20000,"ns":122352453847361},"fail":{"count":0,"ns":0},"queue":{"count":20000,"ns":122035127913357},"compute_input":{"count":20000,"ns":18606312714},"compute_infer":{"count":20000,"ns":297345246187},"compute_output":{"count":20000,"ns":371657389},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2280,"ns":163046500},"compute_infer":{"count":2280,"ns":12790849983},"compute_output":{"count":2280,"ns":27295983}},{"batch_size":2,"compute_input":{"count":3586,"ns":886785550},"compute_infer":{"count":3586,"ns":11497637410},"compute_output":{"count":3586,"ns":47838206}},{"batch_size":4,"compute_input":{"count":183,"ns":106307692},"compute_infer":{"count":183,"ns":11503501280},"compute_output":{"count":183,"ns":3023956}},{"batch_siz

In [144]:
#trying higher request rates 
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --request-count=24000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 24000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 800 inference requests per second
  Client: 
    Request count: 24000
    Throughput: 436.237 infer/sec
    Avg latency: 17143759 usec (standard deviation 15336 usec)
    p50 latency: 17428863 usec
    p90 latency: 19623256 usec
    p95 latency: 20006279 usec
    p99 latency: 20281972 usec
    Avg HTTP time: 17143735 usec (send/recv 37676 usec + response wait 17106059 usec)
  Server: 
    Inference count: 24000
    Execution count: 3003
    Successful request count: 24000
    Avg request latency: 17029848 usec (overhead 80 usec + queue 17015764 usec + compute input 1697 usec + compute infer 12279 usec + compute output 27 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 800, throughput: 436.237 infer/sec, latency 17143759 usec


In [145]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753731227041,"inference_count":24000,"execution_count":3003,"inference_stats":{"success":{"count":24000,"ns":408716353682854},"fail":{"count":0,"ns":0},"queue":{"count":24000,"ns":408378340593111},"compute_input":{"count":24000,"ns":40750811964},"compute_infer":{"count":24000,"ns":294711322680},"compute_output":{"count":24000,"ns":653780707},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":2,"ns":350170},"compute_infer":{"count":2,"ns":6547700154},"compute_output":{"count":2,"ns":44797}},{"batch_size":2,"compute_input":{"count":1,"ns":1715779},"compute_infer":{"count":1,"ns":1451379395},"compute_output":{"count":1,"ns":34179}},{"batch_size":4,"compute_input":{"count":1,"ns":597865},"compute_infer":{"count":1,"ns":11259282698},"compute_output":{"count":1,"ns":43744}},{"batch_size":8,"compute_input":{"count":2999,"ns":50

In [146]:
#run 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --request-count=24000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 24000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 800 inference requests per second
  Client: 
    Request count: 24000
    Throughput: 685.453 infer/sec
    Avg latency: 7948383 usec (standard deviation 14906 usec)
    p50 latency: 7986382 usec
    p90 latency: 10673944 usec
    p95 latency: 11057903 usec
    p99 latency: 11384519 usec
    Avg HTTP time: 7948352 usec (send/recv 11569 usec + response wait 7936783 usec)
  Server: 
    Inference count: 24000
    Execution count: 3017
    Successful request count: 24000
    Avg request latency: 7915106 usec (overhead 83 usec + queue 7904733 usec + compute input 1430 usec + compute infer 8826 usec + compute output 33 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 800, throughput: 685.453 infer/sec, latency 7948383 usec


In [147]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753731405879,"inference_count":48000,"execution_count":6020,"inference_stats":{"success":{"count":48000,"ns":598678918461928},"fail":{"count":0,"ns":0},"queue":{"count":48000,"ns":598091940691161},"compute_input":{"count":48000,"ns":75075049164},"compute_infer":{"count":48000,"ns":506554242155},"compute_output":{"count":48000,"ns":1463364901},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":10,"ns":1022674},"compute_infer":{"count":10,"ns":6571906875},"compute_output":{"count":10,"ns":153839}},{"batch_size":2,"compute_input":{"count":11,"ns":4702699},"compute_infer":{"count":11,"ns":1482141344},"compute_output":{"count":11,"ns":186817}},{"batch_size":4,"compute_input":{"count":5,"ns":4371036},"compute_infer":{"count":5,"ns":11274780496},"compute_output":{"count":5,"ns":127161}},{"batch_size":6,"compute_input":{"count":

In [148]:
#run2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --request-count=24000 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Sending 24000 benchmark requests
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 800 inference requests per second
  Client: 
    Request count: 24000
    Throughput: 799.796 infer/sec
    Avg latency: 16099 usec (standard deviation 4897 usec)
    p50 latency: 15385 usec
    p90 latency: 22754 usec
    p95 latency: 25142 usec
    p99 latency: 29588 usec
    Avg HTTP time: 16064 usec (send/recv 788 usec + response wait 15276 usec)
  Server: 
    Inference count: 24000
    Execution count: 6814
    Successful request count: 24000
    Avg request latency: 12629 usec (overhead 58 usec + queue 7670 usec + compute input 731 usec + compute infer 4147 usec + compute output 23 usec)
Inferences/Second vs. Client Average Batch Latency
Request Rate: 800, throughput: 799.796 infer/sec, latency 16099 usec


In [149]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753731503793,"inference_count":72000,"execution_count":12834,"inference_stats":{"success":{"count":72000,"ns":598982034708749},"fail":{"count":0,"ns":0},"queue":{"count":72000,"ns":598276034474529},"compute_input":{"count":72000,"ns":92622648928},"compute_infer":{"count":72000,"ns":606089472475},"compute_output":{"count":72000,"ns":2017974207},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":184,"ns":15880448},"compute_infer":{"count":184,"ns":7136318967},"compute_output":{"count":184,"ns":2434007}},{"batch_size":2,"compute_input":{"count":3319,"ns":977249492},"compute_infer":{"count":3319,"ns":11899482190},"compute_output":{"count":3319,"ns":52535474}},{"batch_size":4,"compute_input":{"count":1916,"ns":1177329468},"compute_infer":{"count":1916,"ns":18128361282},"compute_output":{"count":1916,"ns":39648842}},{"batch_si

## Test 2: Using time measurement window thing and no tracing (instead of request counts)

In [131]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [133]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 16369
    Avg send request rate: 516.72 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 6.63% of the requests were delayed. 
    Throughput: 511.33 infer/sec
    Avg latency: 14440 usec (standard deviation 23817 usec)
    p50 latency: 9335 usec
    p90 latency: 18435 usec
    p95 latency: 55448 usec
    p99 latency: 91434 usec
    Avg HTTP time: 13999 usec (send/recv 1667 usec + response wait 12332 usec)
  Server: 
    Inference count: 16369
    Execution count: 8604
    Successful request count: 16369
    Avg request latency: 7917 usec (overhead 33 usec + queue 4197 usec + compute input 326 

In [134]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753727100781,"inference_count":62575,"execution_count":16147,"inference_stats":{"success":{"count":62575,"ns":379498276038100},"fail":{"count":0,"ns":0},"queue":{"count":62575,"ns":378855245194208},"compute_input":{"count":62575,"ns":81806505563},"compute_infer":{"count":62575,"ns":555324752924},"compute_output":{"count":62575,"ns":1626752114},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":3407,"ns":326803613},"compute_infer":{"count":3407,"ns":17229888724},"compute_output":{"count":3407,"ns":41598892}},{"batch_size":2,"compute_input":{"count":6667,"ns":1663427957},"compute_infer":{"count":6667,"ns":22632126952},"compute_output":{"count":6667,"ns":91446035}},{"batch_size":4,"compute_input":{"count":623,"ns":345834663},"compute_infer":{"count":623,"ns":14590230600},"compute_output":{"count":623,"ns":11140735}},{"batch

In [135]:
#test 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 13007
    Avg send request rate: 462.49 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.14% of the requests were delayed. 
    Throughput: 500.23 infer/sec
    Avg latency: 9820 usec (standard deviation 10219 usec)
    p50 latency: 8552 usec
    p90 latency: 13284 usec
    p95 latency: 16717 usec
    p99 latency: 34667 usec
    Avg HTTP time: 9568 usec (send/recv 578 usec + response wait 8990 usec)
  Server: 
    Inference count: 13006
    Execution count: 7131
    Successful request count: 13006
    Avg request latency: 6907 usec (overhead 30 usec + queue 3410 usec + compute input 276 usec

In [136]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753727197774,"inference_count":97826,"execution_count":35093,"inference_stats":{"success":{"count":97826,"ns":379772879171795},"fail":{"count":0,"ns":0},"queue":{"count":97826,"ns":379001607950629},"compute_input":{"count":97826,"ns":92983060105},"compute_infer":{"count":97826,"ns":670769165540},"compute_output":{"count":97826,"ns":2155142251},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":9610,"ns":888054875},"compute_infer":{"count":9610,"ns":36363983708},"compute_output":{"count":9610,"ns":116850515}},{"batch_size":2,"compute_input":{"count":18170,"ns":4445158423},"compute_infer":{"count":18170,"ns":57551770633},"compute_output":{"count":18170,"ns":247343480}},{"batch_size":4,"compute_input":{"count":1516,"ns":857201030},"compute_infer":{"count":1516,"ns":17790424856},"compute_output":{"count":1516,"ns":27072304}}

In [137]:
#test 2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 14323
    Avg send request rate: 504.08 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 4.81% of the requests were delayed. 
    Throughput: 509.01 infer/sec
    Avg latency: 11241 usec (standard deviation 16044 usec)
    p50 latency: 8431 usec
    p90 latency: 14559 usec
    p95 latency: 28877 usec
    p99 latency: 62237 usec
    Avg HTTP time: 10952 usec (send/recv 714 usec + response wait 10238 usec)
  Server: 
    Inference count: 14323
    Execution count: 7836
    Successful request count: 14323
    Avg request latency: 7117 usec (overhead 30 usec + queue 3613 usec + compute input 281 u

In [138]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753727351724,"inference_count":148076,"execution_count":61487,"inference_stats":{"success":{"count":148076,"ns":380173603413311},"fail":{"count":0,"ns":0},"queue":{"count":148076,"ns":379214473625443},"compute_input":{"count":148076,"ns":108959845600},"compute_infer":{"count":148076,"ns":840347432367},"compute_output":{"count":148076,"ns":2895266082},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":17774,"ns":1634321824},"compute_infer":{"count":17774,"ns":62068707713},"compute_output":{"count":17774,"ns":214945488}},{"batch_size":2,"compute_input":{"count":34381,"ns":8307187461},"compute_infer":{"count":34381,"ns":108459777291},"compute_output":{"count":34381,"ns":466227774}},{"batch_size":4,"compute_input":{"count":2996,"ns":1683377605},"compute_infer":{"count":2996,"ns":23233261524},"compute_output":{"count":2996,"n

In [139]:
#test 3
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 500 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 500 inference requests per second
  Client: 
    Request count: 13156
    Avg send request rate: 519.57 infer/sec
    [WARNING] Perf Analyzer was not able to keep up with the desired request rate. 3.62% of the requests were delayed. 
    Throughput: 511.62 infer/sec
    Avg latency: 12721 usec (standard deviation 12369 usec)
    p50 latency: 10895 usec
    p90 latency: 16049 usec
    p95 latency: 22267 usec
    p99 latency: 61739 usec
    Avg HTTP time: 12540 usec (send/recv 608 usec + response wait 11932 usec)
  Server: 
    Inference count: 13158
    Execution count: 6243
    Successful request count: 13158
    Avg request latency: 9045 usec (overhead 33 usec + queue 4898 usec + compute input 330 

In [140]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":1753727639909,"inference_count":195153,"execution_count":85744,"inference_stats":{"success":{"count":195153,"ns":380554830927364},"fail":{"count":0,"ns":0},"queue":{"count":195153,"ns":379415917669771},"compute_input":{"count":195153,"ns":123865520007},"compute_infer":{"count":195153,"ns":1003053082908},"compute_output":{"count":195153,"ns":3596758931},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[{"batch_size":1,"compute_input":{"count":24639,"ns":2259781673},"compute_infer":{"count":24639,"ns":84399981120},"compute_output":{"count":24639,"ns":299667483}},{"batch_size":2,"compute_input":{"count":49745,"ns":11969336462},"compute_infer":{"count":49745,"ns":158396379897},"compute_output":{"count":49745,"ns":677962082}},{"batch_size":4,"compute_input":{"count":4539,"ns":2526075488},"compute_infer":{"count":4539,"ns":28935796552},"compute_output":{"count":4539,

Now let's try higher arrival rates 

In [141]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

{"model_stats":[{"name":"food_classifier_onnx","version":"1","last_inference":0,"inference_count":0,"execution_count":0,"inference_stats":{"success":{"count":0,"ns":0},"fail":{"count":0,"ns":0},"queue":{"count":0,"ns":0},"compute_input":{"count":0,"ns":0},"compute_infer":{"count":0,"ns":0},"compute_output":{"count":0,"ns":0},"cache_hit":{"count":0,"ns":0},"cache_miss":{"count":0,"ns":0}},"response_stats":{},"batch_stats":[],"memory_usage":[]}]}


In [143]:
#warmup
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --async

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 800 inference requests per second
Failed to retrieve results from inference request.
Thread [0] had error: failed to parse the request JSON buffer: The document is empty. at 0

Thread [1] had error: failed to parse the request JSON buffer: The document is empty. at 0

Thread [2] had error: failed to parse the request JSON buffer: The document is empty. at 0

Thread [3] had error: failed to parse the request JSON buffer: The document is empty. at 0




: 99

In [ ]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

In [ ]:
#run 1
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --async

In [ ]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats

In [ ]:
#run 2
perf_analyzer -u triton_server:8000  -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 \
    --request-distribution=poisson --request-rate-range 800 --async

In [ ]:
curl http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats